In [ ]:
!nvidia-smi

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("Multiprocessor count:", torch.cuda.get_device_properties(0).multi_processor_count)
print("Total memory (MB):", torch.cuda.get_device_properties(0).total_memory // (1024**2))
print("Max threads per block:", torch.cuda.get_device_properties(0).max_threads_per_block)
print("Max threads dimensions:", torch.cuda.get_device_properties(0).max_threads_dim)
print("Max grid size:", torch.cuda.get_device_properties(0).max_grid_size)


In [ ]:
!nvidia-smi -q


In [ ]:
from google.colab import files
files.upload()

# Code initial

In [ ]:
!ls


In [ ]:
%%writefile neural_net.cu

// Parallel matrix multiplication using CUDA
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE      32
#define HIDDEN_SIZE     256
#define OUTPUT_SIZE     1
#define EPOCHS          100
#define LOG_EVERY_EPOCH 1
#define LEARNING_RATE   0.01
#define BATCH_SIZE      128
#define THREADS_PER_BLOCK 16

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

// ---------------- Memory Management ----------------
Matrix* allocate_matrix(int rows, int cols) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = rows;
    m->cols = cols;
    m->data = (float*)malloc(rows * cols * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    free(m->data);
    free(m);
}

// ---------------- Matrix Operations ----------------
void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows; i++)
        for (int j = 0; j < m->cols; j++)
            m->data[i * m->cols + j] = ((float)rand() / RAND_MAX * 2.0f - 1.0f) * limit;
}

// ---------------- CUDA Kernel ----------------
__global__ void mat_mult_kernel(float *A, float *B, float *C, int A_rows, int A_cols, int B_cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < A_rows && col < B_cols) {
        float value = 0.0f;
        for (int k = 0; k < A_cols; k++) {
            value += A[row * A_cols + k] * B[k * B_cols + col];
        }
        C[row * B_cols + col] = value;
    }
}

// ---------------- GPU Matrix Multiplication ----------------
Matrix* mat_mult_gpu(Matrix *A, Matrix *B) {
    if (A->cols != B->rows) {
        printf("Incompatible matrices for multiplication.\n");
        exit(1);
    }

    Matrix *C = allocate_matrix(A->rows, B->cols);
    float *d_A, *d_B, *d_C;
    size_t sizeA = A->rows * A->cols * sizeof(float);
    size_t sizeB = B->rows * B->cols * sizeof(float);
    size_t sizeC = C->rows * C->cols * sizeof(float);

    cudaMalloc((void **)&d_A, sizeA);
    cudaMalloc((void **)&d_B, sizeB);
    cudaMalloc((void **)&d_C, sizeC);

    cudaMemcpy(d_A, A->data, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B->data, sizeB, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(THREADS_PER_BLOCK, THREADS_PER_BLOCK);
    dim3 numBlocks((B->cols + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (A->rows + threadsPerBlock.y - 1) / threadsPerBlock.y);

    mat_mult_kernel<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, A->rows, A->cols, B->cols);
    cudaDeviceSynchronize();

    cudaMemcpy(C->data, d_C, sizeC, cudaMemcpyDeviceToHost);

    // Verification erreur CUDA
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("CUDA Error: %s\n", cudaGetErrorString(err));
        exit(1);
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return C;
}

// ---------------- CPU Functions ----------------
Matrix* mat_sub(Matrix *A, Matrix *B) {
    if(A->rows != B->rows || A->cols != B->cols) {
        printf("Incompatible matrices for subtraction.\n");
        exit(1);
    }
    Matrix *C = allocate_matrix(A->rows, A->cols);
    for(int i = 0; i < A->rows; i++)
        for(int j = 0; j < A->cols; j++)
            C->data[i * A->cols + j] = A->data[i * A->cols + j] - B->data[i * A->cols + j];
    return C;
}

void mat_scalar_mult(Matrix *A, float scalar) {
    for(int i = 0; i < A->rows; i++)
        for(int j = 0; j < A->cols; j++)
            A->data[i * A->cols + j] *= scalar;
}

void relu(Matrix *m) {
    for(int i = 0; i < m->rows; i++)
        for(int j = 0; j < m->cols; j++)
            m->data[i * m->cols + j] = fmaxf(0, m->data[i * m->cols + j]);
}

Matrix* relu_derivative(Matrix *m) {
    Matrix *derivative = allocate_matrix(m->rows, m->cols);
    for(int i = 0; i < m->rows; i++)
        for(int j = 0; j < m->cols; j++)
            derivative->data[i * m->cols + j] = (m->data[i * m->cols + j] > 0) ? 1 : 0;
    return derivative;
}

float mean_squared_error(Matrix *Y_pred, Matrix *Y_true) {
    float mse = 0.0f;
    for(int i = 0; i < Y_pred->rows; i++)
        for(int j = 0; j < Y_pred->cols; j++)
            mse += pow(Y_pred->data[i * Y_pred->cols + j] - Y_true->data[i * Y_true->cols + j], 2);
    return mse / Y_pred->rows;
}

void update_weights(Matrix *W, Matrix *grad, float learning_rate) {
    for(int i = 0; i < W->rows; i++)
        for(int j = 0; j < W->cols; j++)
            W->data[i * W->cols + j] -= learning_rate * grad->data[i * grad->cols + j];
}

void update_bias(Matrix *b, Matrix *grad, float learning_rate, int batch_size) {
    for(int j = 0; j < b->cols; j++) {
        float sum = 0.0f;
        for(int i = 0; i < grad->rows; i++) {
            sum += grad->data[i * grad->cols + j];
        }
        b->data[j] -= learning_rate * sum / batch_size;
    }
}

void backpropagation(Matrix *X_batch, Matrix *Y_batch, Matrix *Z1, Matrix *Y_pred,
                     Matrix *W1, Matrix *W2, Matrix *b1, Matrix *b2, int batch_size) {
    // Gradient sortie
    Matrix *dZ2 = mat_sub(Y_pred, Y_batch);
    mat_scalar_mult(dZ2, 2.0f / batch_size);

    // Gradient W2
    Matrix *Z1_T = allocate_matrix(Z1->cols, Z1->rows);
    for(int i = 0; i < Z1->rows; i++)
        for(int j = 0; j < Z1->cols; j++)
            Z1_T->data[j * Z1->rows + i] = Z1->data[i * Z1->cols + j];
    Matrix *dW2 = mat_mult_gpu(Z1_T, dZ2);
    update_weights(W2, dW2, LEARNING_RATE);
    update_bias(b2, dZ2, LEARNING_RATE, batch_size);
    free_matrix(dW2);
    free_matrix(Z1_T);

    // Gradient couche cachée
    Matrix *W2_T = allocate_matrix(W2->cols, W2->rows);
    for(int i = 0; i < W2->rows; i++)
        for(int j = 0; j < W2->cols; j++)
            W2_T->data[j * W2->rows + i] = W2->data[i * W2->cols + j];
    Matrix *dZ1 = mat_mult_gpu(dZ2, W2_T);

    Matrix *dZ1_derivative = relu_derivative(Z1);
    for(int i = 0; i < dZ1->rows; i++)
        for(int j = 0; j < dZ1->cols; j++)
            dZ1->data[i * dZ1->cols + j] *= dZ1_derivative->data[i * dZ1_derivative->cols + j];
    free_matrix(dZ1_derivative);
    free_matrix(W2_T);

    // Gradient W1
    Matrix *X_batch_T = allocate_matrix(X_batch->cols, X_batch->rows);
    for(int i = 0; i < X_batch->rows; i++)
        for(int j = 0; j < X_batch->cols; j++)
            X_batch_T->data[j * X_batch->rows + i] = X_batch->data[i * X_batch->cols + j];
    Matrix *dW1 = mat_mult_gpu(X_batch_T, dZ1);
    update_weights(W1, dW1, LEARNING_RATE);
    update_bias(b1, dZ1, LEARNING_RATE, batch_size);
    free_matrix(dW1);
    free_matrix(X_batch_T);

    free_matrix(dZ2);
    free_matrix(dZ1);
}

void get_batch(Matrix *X, Matrix *Y, Matrix *X_batch, Matrix *Y_batch, int batch_start, int batch_size) {
    for(int i = 0; i < batch_size; i++) {
        for(int j = 0; j < INPUT_SIZE; j++)
            X_batch->data[i * INPUT_SIZE + j] = X->data[(batch_start + i) * INPUT_SIZE + j];
        Y_batch->data[i * OUTPUT_SIZE] = Y->data[(batch_start + i) * OUTPUT_SIZE];
    }
}

int load_csv(const char *filename, Matrix **X, Matrix **Y, int *num_samples) {
    FILE *file = fopen(filename, "r");
    if(!file) {
        printf("Failed to open file %s\n", filename);
        return -1;
    }
    char line[1024];
    int count = 0;
    while(fgets(line, sizeof(line), file)) count++;
    *num_samples = count;
    rewind(file);

    *X = allocate_matrix(count, INPUT_SIZE);
    *Y = allocate_matrix(count, OUTPUT_SIZE);

    int i = 0;
    while(fgets(line, sizeof(line), file)) {
        char *token = strtok(line, ",");
        int j = 0;
        while(token) {
            if(j < INPUT_SIZE)
                (*X)->data[i * INPUT_SIZE + j] = atof(token);
            else
                (*Y)->data[i * OUTPUT_SIZE] = atof(token);
            token = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(file);
    return 0;
}

// ---------------- Main Function ----------------
// ---------------- Main Function ----------------
int main() {
    srand(time(NULL));

    // Chargement dataset
    int num_samples = 512;
    Matrix *X = NULL;
    Matrix *Y = NULL;

    if (load_csv("synthetic_convex_small.csv", &X, &Y, &num_samples) != 0) {
        printf("Erreur chargement dataset\n");
        return -1;
    }

    printf("Dataset charge depuis fichier (%d samples)\n", num_samples);

    // Vérification batch size
    if (BATCH_SIZE >= num_samples) {
        printf("ERREUR: BATCH_SIZE (%d) doit etre < num_samples (%d)\n", BATCH_SIZE, num_samples);
        free_matrix(X);
        free_matrix(Y);
        return -1;
    }

    // Initialisation poids et biais
    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    Matrix *b1 = allocate_matrix(1, HIDDEN_SIZE);
    Matrix *b2 = allocate_matrix(1, OUTPUT_SIZE);
    memset(b1->data, 0, HIDDEN_SIZE * sizeof(float));
    memset(b2->data, 0, OUTPUT_SIZE * sizeof(float));

    int effective_batch_size = (num_samples < BATCH_SIZE) ? num_samples : BATCH_SIZE;
    Matrix *X_batch = allocate_matrix(effective_batch_size, INPUT_SIZE);
    Matrix *Y_batch = allocate_matrix(effective_batch_size, OUTPUT_SIZE);

    // Mesure temps total global
    cudaEvent_t start_total, stop_total;
    cudaEventCreate(&start_total);
    cudaEventCreate(&stop_total);
    cudaEventRecord(start_total);

    // Variables cumulatives pour chaque étape
    float total_time_forward = 0.0f;
    float total_time_activation = 0.0f;
    float total_time_output = 0.0f;
    float total_time_backprop = 0.0f;

    int num_batches = num_samples / effective_batch_size;
    for(int epoch = 0; epoch < EPOCHS; epoch++) {
        float epoch_loss = 0.0f;

        for(int batch_idx = 0; batch_idx < num_batches; batch_idx++) {
            int batch_start = batch_idx * effective_batch_size;
            get_batch(X, Y, X_batch, Y_batch, batch_start, effective_batch_size);

            // CUDA events pour batch
            cudaEvent_t start_fp, stop_fp;
            cudaEvent_t start_act, stop_act;
            cudaEvent_t start_out, stop_out;
            cudaEvent_t start_bp, stop_bp;

            cudaEventCreate(&start_fp); cudaEventCreate(&stop_fp);
            cudaEventCreate(&start_act); cudaEventCreate(&stop_act);
            cudaEventCreate(&start_out); cudaEventCreate(&stop_out);
            cudaEventCreate(&start_bp); cudaEventCreate(&stop_bp);

            // ---------------- Forward couche cachée ----------------
            cudaEventRecord(start_fp);
            Matrix *Z1 = mat_mult_gpu(X_batch, W1);
            cudaEventRecord(stop_fp);

            // Activation ReLU
            cudaEventRecord(start_act);
            for(int i = 0; i < Z1->rows; i++)
                for(int j = 0; j < Z1->cols; j++)
                    Z1->data[i * Z1->cols + j] += b1->data[j];
            relu(Z1);
            cudaEventRecord(stop_act);

            // Forward couche sortie
            cudaEventRecord(start_out);
            Matrix *Y_pred = mat_mult_gpu(Z1, W2);
            for(int i = 0; i < Y_pred->rows; i++)
                for(int j = 0; j < Y_pred->cols; j++)
                    Y_pred->data[i * Y_pred->cols + j] += b2->data[j];
            cudaEventRecord(stop_out);

            float batch_loss = mean_squared_error(Y_pred, Y_batch);
            epoch_loss += batch_loss;

            // ---------------- Backpropagation ----------------
            cudaEventRecord(start_bp);
            backpropagation(X_batch, Y_batch, Z1, Y_pred, W1, W2, b1, b2, effective_batch_size);
            cudaEventRecord(stop_bp);

            free_matrix(Z1);
            free_matrix(Y_pred);

            // Cumuler les temps pour toutes les étapes
            float t_fp, t_act, t_out, t_bp;
            cudaEventElapsedTime(&t_fp, start_fp, stop_fp);
            cudaEventElapsedTime(&t_act, start_act, stop_act);
            cudaEventElapsedTime(&t_out, start_out, stop_out);
            cudaEventElapsedTime(&t_bp, start_bp, stop_bp);

            total_time_forward += t_fp;
            total_time_activation += t_act;
            total_time_output += t_out;
            total_time_backprop += t_bp;

            // Détruire events batch
            cudaEventDestroy(start_fp); cudaEventDestroy(stop_fp);
            cudaEventDestroy(start_act); cudaEventDestroy(stop_act);
            cudaEventDestroy(start_out); cudaEventDestroy(stop_out);
            cudaEventDestroy(start_bp); cudaEventDestroy(stop_bp);
        }

        epoch_loss /= num_batches;
        if(epoch % 10 == 0)
            printf("Epoch %d, MSE moyenne: %f\n", epoch, epoch_loss);
    }

    cudaEventRecord(stop_total);
    cudaEventSynchronize(stop_total);
    float total_time = 0;
    cudaEventElapsedTime(&total_time, start_total, stop_total);

    // Affichage temps cumulé par étape
    printf("\nTemps cumulé Forward: %f ms\n", total_time_forward);
    printf("Temps cumulé Activation: %f ms\n", total_time_activation);
    printf("Temps cumulé Output: %f ms\n", total_time_output);
    printf("Temps cumulé Backprop: %f ms\n", total_time_backprop);
    printf("Temps total training: %f ms (%.4f s)\n", total_time, total_time/1000.0f);

    // Cleanup
    cudaEventDestroy(start_total); cudaEventDestroy(stop_total);
    free_matrix(X_batch); free_matrix(Y_batch);
    free_matrix(W1); free_matrix(W2); free_matrix(b1); free_matrix(b2);
    free_matrix(X); free_matrix(Y);

    return 0;
}


Etape 0: B transpose


In [ ]:
%%writefile neural_net_BT.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE      32
#define HIDDEN_SIZE     256
#define OUTPUT_SIZE     1
#define EPOCHS          100
#define LEARNING_RATE   0.01f
#define BATCH_SIZE      128
#define THREADS_PER_BLOCK 16

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

/* ================= MEMORY MANAGEMENT ================= */

Matrix* allocate_matrix(int rows, int cols) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = rows;
    m->cols = cols;
    m->data = (float*)malloc(rows * cols * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    free(m->data);
    free(m);
}

/* ================= INITIALIZATION ================= */

void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = ((float)rand() / RAND_MAX * 2.0f - 1.0f) * limit;
}

/* ================= MATRIX TRANSPOSE (CPU) ================= */

Matrix* transpose_matrix(Matrix *B) {
    Matrix *B_T = allocate_matrix(B->cols, B->rows);
    for (int i = 0; i < B->rows; i++)
        for (int j = 0; j < B->cols; j++)
            B_T->data[j * B->rows + i] = B->data[i * B->cols + j];
    return B_T;
}

/* ================= CUDA KERNEL (A × Bᵀ) ================= */

__global__ void mat_mult_BT_kernel(float *A, float *B_T, float *C,
                                  int A_rows, int A_cols, int B_cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < A_rows && col < B_cols) {
        float sum = 0.0f;
        for (int k = 0; k < A_cols; k++)
            sum += A[row * A_cols + k] * B_T[col * A_cols + k];
        C[row * B_cols + col] = sum;
    }
}

/* ================= GPU MATMUL WITH B TRANSPOSE ================= */

Matrix* mat_mult_gpu_BT(Matrix *A, Matrix *B) {
    if (A->cols != B->rows) {
        printf("Matrix size mismatch\n");
        exit(1);
    }

    Matrix *B_T = transpose_matrix(B);
    Matrix *C   = allocate_matrix(A->rows, B->cols);

    float *d_A, *d_B_T, *d_C;
    size_t sizeA = A->rows * A->cols * sizeof(float);
    size_t sizeB = B_T->rows * B_T->cols * sizeof(float);
    size_t sizeC = C->rows * C->cols * sizeof(float);

    cudaMalloc(&d_A, sizeA);
    cudaMalloc(&d_B_T, sizeB);
    cudaMalloc(&d_C, sizeC);

    cudaMemcpy(d_A, A->data, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B_T, B_T->data, sizeB, cudaMemcpyHostToDevice);

    dim3 block(THREADS_PER_BLOCK, THREADS_PER_BLOCK);
    dim3 grid((C->cols + block.x - 1) / block.x,
              (C->rows + block.y - 1) / block.y);

    mat_mult_BT_kernel<<<grid, block>>>(d_A, d_B_T, d_C,
                                       A->rows, A->cols, C->cols);
    cudaDeviceSynchronize();

    cudaMemcpy(C->data, d_C, sizeC, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B_T);
    cudaFree(d_C);
    free_matrix(B_T);

    return C;
}

/* ================= CPU OPS ================= */

void relu(Matrix *m) {
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = fmaxf(0.0f, m->data[i]);
}

Matrix* relu_derivative(Matrix *m) {
    Matrix *d = allocate_matrix(m->rows, m->cols);
    for (int i = 0; i < m->rows * m->cols; i++)
        d->data[i] = (m->data[i] > 0) ? 1.0f : 0.0f;
    return d;
}

Matrix* mat_sub(Matrix *A, Matrix *B) {
    Matrix *C = allocate_matrix(A->rows, A->cols);
    for (int i = 0; i < A->rows * A->cols; i++)
        C->data[i] = A->data[i] - B->data[i];
    return C;
}

void mat_scalar_mult(Matrix *A, float s) {
    for (int i = 0; i < A->rows * A->cols; i++)
        A->data[i] *= s;
}

float mean_squared_error(Matrix *Yp, Matrix *Y) {
    float mse = 0.0f;
    for (int i = 0; i < Yp->rows; i++)
        mse += powf(Yp->data[i] - Y->data[i], 2);
    return mse / Yp->rows;
}

/* ================= BACKPROP ================= */

void backpropagation(Matrix *X, Matrix *Y, Matrix *Z1, Matrix *Yp,
                     Matrix *W1, Matrix *W2, Matrix *b1, Matrix *b2, int bs) {

    Matrix *dZ2 = mat_sub(Yp, Y);
    mat_scalar_mult(dZ2, 2.0f / bs);

    Matrix *Z1_T = transpose_matrix(Z1);
    Matrix *dW2 = mat_mult_gpu_BT(Z1_T, dZ2);
    free_matrix(Z1_T);

    Matrix *W2_T = transpose_matrix(W2);
    Matrix *dZ1 = mat_mult_gpu_BT(dZ2, W2_T);
    free_matrix(W2_T);

    Matrix *relu_d = relu_derivative(Z1);
    for (int i = 0; i < dZ1->rows * dZ1->cols; i++)
        dZ1->data[i] *= relu_d->data[i];
    free_matrix(relu_d);

    Matrix *X_T = transpose_matrix(X);
    Matrix *dW1 = mat_mult_gpu_BT(X_T, dZ1);
    free_matrix(X_T);

    free_matrix(dW1);
    free_matrix(dW2);
    free_matrix(dZ1);
    free_matrix(dZ2);
}

/* ================= BATCH FUNCTIONS ================= */

void get_batch(Matrix *X, Matrix *Y, Matrix *X_batch, Matrix *Y_batch, int batch_start, int batch_size) {
    for(int i = 0; i < batch_size; i++) {
        for(int j = 0; j < INPUT_SIZE; j++)
            X_batch->data[i * INPUT_SIZE + j] = X->data[(batch_start + i) * INPUT_SIZE + j];
        Y_batch->data[i * OUTPUT_SIZE] = Y->data[(batch_start + i) * OUTPUT_SIZE];
    }
}

/* ================= LOAD CSV ================= */

int load_csv(const char *filename, Matrix **X, Matrix **Y, int *num_samples) {
    FILE *file = fopen(filename, "r");
    if(!file) {
        printf("Failed to open file %s\n", filename);
        return -1;
    }
    char line[1024];
    int count = 0;
    while(fgets(line, sizeof(line), file)) count++;
    *num_samples = count;
    rewind(file);

    *X = allocate_matrix(count, INPUT_SIZE);
    *Y = allocate_matrix(count, OUTPUT_SIZE);

    int i = 0;
    while(fgets(line, sizeof(line), file)) {
        char *token = strtok(line, ",");
        int j = 0;
        while(token) {
            if(j < INPUT_SIZE)
                (*X)->data[i * INPUT_SIZE + j] = atof(token);
            else
                (*Y)->data[i * OUTPUT_SIZE] = atof(token);
            token = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(file);
    return 0;
}

/* ================= MAIN ================= */

int main() {
    srand(time(NULL));

    int num_samples = 512;
    Matrix *X = NULL;
    Matrix *Y = NULL;

    if(load_csv("synthetic_convex_small.csv", &X, &Y, &num_samples) != 0) {
        printf("Erreur chargement dataset\n");
        return -1;
    }

    printf("Dataset charge depuis fichier (%d samples)\n", num_samples);

    if(BATCH_SIZE >= num_samples) {
        printf("ERREUR: BATCH_SIZE (%d) doit etre < num_samples (%d)\n", BATCH_SIZE, num_samples);
        free_matrix(X); free_matrix(Y);
        return -1;
    }

    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    Matrix *b1 = allocate_matrix(1, HIDDEN_SIZE);
    Matrix *b2 = allocate_matrix(1, OUTPUT_SIZE);
    memset(b1->data, 0, HIDDEN_SIZE * sizeof(float));
    memset(b2->data, 0, OUTPUT_SIZE * sizeof(float));

    int effective_batch_size = (num_samples < BATCH_SIZE) ? num_samples : BATCH_SIZE;
    Matrix *X_batch = allocate_matrix(effective_batch_size, INPUT_SIZE);
    Matrix *Y_batch = allocate_matrix(effective_batch_size, OUTPUT_SIZE);

    cudaEvent_t start_total, stop_total;
    cudaEventCreate(&start_total);
    cudaEventCreate(&stop_total);
    cudaEventRecord(start_total);

    float total_time_forward = 0.0f, total_time_activation = 0.0f;
    float total_time_output = 0.0f, total_time_backprop = 0.0f;

    int num_batches = num_samples / effective_batch_size;

    for(int epoch = 0; epoch < EPOCHS; epoch++) {
        for(int batch_idx = 0; batch_idx < num_batches; batch_idx++) {
            int batch_start = batch_idx * effective_batch_size;
            get_batch(X, Y, X_batch, Y_batch, batch_start, effective_batch_size);

            cudaEvent_t start_fp, stop_fp, start_act, stop_act, start_out, stop_out, start_bp, stop_bp;
            cudaEventCreate(&start_fp); cudaEventCreate(&stop_fp);
            cudaEventCreate(&start_act); cudaEventCreate(&stop_act);
            cudaEventCreate(&start_out); cudaEventCreate(&stop_out);
            cudaEventCreate(&start_bp); cudaEventCreate(&stop_bp);

            cudaEventRecord(start_fp);
            Matrix *Z1 = mat_mult_gpu_BT(X_batch, W1);
            cudaEventRecord(stop_fp);

            cudaEventRecord(start_act);
            for(int i = 0; i < Z1->rows; i++)
                for(int j = 0; j < Z1->cols; j++)
                    Z1->data[i * Z1->cols + j] += b1->data[j];
            relu(Z1);
            cudaEventRecord(stop_act);

            cudaEventRecord(start_out);
            Matrix *Y_pred = mat_mult_gpu_BT(Z1, W2);
            for(int i = 0; i < Y_pred->rows; i++)
                for(int j = 0; j < Y_pred->cols; j++)
                    Y_pred->data[i * Y_pred->cols + j] += b2->data[j];
            cudaEventRecord(stop_out);

            cudaEventRecord(start_bp);
            backpropagation(X_batch, Y_batch, Z1, Y_pred, W1, W2, b1, b2, effective_batch_size);
            cudaEventRecord(stop_bp);

            free_matrix(Z1); free_matrix(Y_pred);

            float t_fp, t_act, t_out, t_bp;
            cudaEventElapsedTime(&t_fp, start_fp, stop_fp);
            cudaEventElapsedTime(&t_act, start_act, stop_act);
            cudaEventElapsedTime(&t_out, start_out, stop_out);
            cudaEventElapsedTime(&t_bp, start_bp, stop_bp);

            total_time_forward += t_fp;
            total_time_activation += t_act;
            total_time_output += t_out;
            total_time_backprop += t_bp;

            cudaEventDestroy(start_fp); cudaEventDestroy(stop_fp);
            cudaEventDestroy(start_act); cudaEventDestroy(stop_act);
            cudaEventDestroy(start_out); cudaEventDestroy(stop_out);
            cudaEventDestroy(start_bp); cudaEventDestroy(stop_bp);
        }
    }

    cudaEventRecord(stop_total);
    cudaEventSynchronize(stop_total);
    float total_time = 0.0f;
    cudaEventElapsedTime(&total_time, start_total, stop_total);

    printf("\nTemps cumulé Forward    : %f ms\n", total_time_forward);
    printf("Temps cumulé Activation : %f ms\n", total_time_activation);
    printf("Temps cumulé Output     : %f ms\n", total_time_output);
    printf("Temps cumulé Backprop   : %f ms\n", total_time_backprop);
    printf("Temps total training    : %f ms (%.4f s)\n", total_time, total_time/1000.0f);

    cudaEventDestroy(start_total); cudaEventDestroy(stop_total);
    free_matrix(X_batch); free_matrix(Y_batch);
    free_matrix(W1); free_matrix(W2); free_matrix(b1); free_matrix(b2);
    free_matrix(X); free_matrix(Y);

    return 0;
}


In [ ]:
!nvcc -O3 -lineinfo neural_net_BT.cu -o neural_net_BT


In [ ]:
!./neural_net_BT


# Étape 1 — Optimisation par Shared Memory (Tiled Matrix Multiplication)


In [ ]:
%%writefile neural_net_BT_T.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE 32
#define HIDDEN_SIZE 256
#define OUTPUT_SIZE 1
#define EPOCHS 100
#define LEARNING_RATE 0.01f
#define BATCH_SIZE 128
#define TILE 16

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

/* ================= MEMORY ================= */

Matrix* allocate_matrix(int r, int c) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = r;
    m->cols = c;
    m->data = (float*)malloc(r * c * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    free(m->data);
    free(m);
}

/* ================= INIT ================= */

void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = ((float)rand() / RAND_MAX * 2 - 1) * limit;
}

/* ================= TRANSPOSE ================= */

Matrix* transpose_matrix(Matrix *B) {
    Matrix *T = allocate_matrix(B->cols, B->rows);
    for (int i = 0; i < B->rows; i++)
        for (int j = 0; j < B->cols; j++)
            T->data[j * B->rows + i] = B->data[i * B->cols + j];
    return T;
}

/* ================= SHARED MEMORY KERNEL ================= */

__global__ void matmul_BT_shared(float *A, float *B_T, float *C,
                                 int A_rows, int A_cols, int B_cols) {

    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float sum = 0.0f;

    for (int t = 0; t < (A_cols + TILE - 1) / TILE; t++) {

        int a_col = t * TILE + threadIdx.x;
        int b_col = t * TILE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] =
            (row < A_rows && a_col < A_cols)
            ? A[row * A_cols + a_col] : 0.0f;

        Bs[threadIdx.y][threadIdx.x] =
            (col < B_cols && b_col < A_cols)
            ? B_T[col * A_cols + b_col] : 0.0f;

        __syncthreads();

        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];

        __syncthreads();
    }

    if (row < A_rows && col < B_cols)
        C[row * B_cols + col] = sum;
}

/* ================= GPU MATMUL ================= */

Matrix* mat_mult_gpu_BT(Matrix *A, Matrix *B) {
    Matrix *B_T = transpose_matrix(B);
    Matrix *C = allocate_matrix(A->rows, B->cols);

    float *dA, *dB, *dC;
    cudaMalloc(&dA, A->rows * A->cols * sizeof(float));
    cudaMalloc(&dB, B_T->rows * B_T->cols * sizeof(float));
    cudaMalloc(&dC, C->rows * C->cols * sizeof(float));

    cudaMemcpy(dA, A->data, A->rows * A->cols * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B_T->data, B_T->rows * B_T->cols * sizeof(float), cudaMemcpyHostToDevice);

    dim3 block(TILE, TILE);
    dim3 grid((C->cols + TILE - 1) / TILE,
              (C->rows + TILE - 1) / TILE);

    matmul_BT_shared<<<grid, block>>>(dA, dB, dC,
                                     A->rows, A->cols, C->cols);
    cudaDeviceSynchronize();

    cudaMemcpy(C->data, dC, C->rows * C->cols * sizeof(float), cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free_matrix(B_T);

    return C;
}

/* ================= CPU OPS ================= */

void relu(Matrix *m) {
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = fmaxf(0.0f, m->data[i]);
}

Matrix* relu_derivative(Matrix *m) {
    Matrix *d = allocate_matrix(m->rows, m->cols);
    for (int i = 0; i < m->rows * m->cols; i++)
        d->data[i] = (m->data[i] > 0);
    return d;
}

Matrix* mat_sub(Matrix *A, Matrix *B) {
    Matrix *C = allocate_matrix(A->rows, A->cols);
    for (int i = 0; i < A->rows * A->cols; i++)
        C->data[i] = A->data[i] - B->data[i];
    return C;
}

void mat_scalar_mult(Matrix *A, float s) {
    for (int i = 0; i < A->rows * A->cols; i++)
        A->data[i] *= s;
}

/* ================= BACKPROP ================= */

void backpropagation(Matrix *X, Matrix *Y, Matrix *Z1, Matrix *Yp,
                     Matrix *W1, Matrix *W2, int bs) {

    Matrix *dZ2 = mat_sub(Yp, Y);
    mat_scalar_mult(dZ2, 2.0f / bs);

    Matrix *Z1_T = transpose_matrix(Z1);
    free_matrix(mat_mult_gpu_BT(Z1_T, dZ2));
    free_matrix(Z1_T);

    Matrix *W2_T = transpose_matrix(W2);
    Matrix *dZ1 = mat_mult_gpu_BT(dZ2, W2_T);
    free_matrix(W2_T);

    Matrix *relu_d = relu_derivative(Z1);
    for (int i = 0; i < dZ1->rows * dZ1->cols; i++)
        dZ1->data[i] *= relu_d->data[i];
    free_matrix(relu_d);

    Matrix *X_T = transpose_matrix(X);
    free_matrix(mat_mult_gpu_BT(X_T, dZ1));

    free_matrix(X_T);
    free_matrix(dZ1);
    free_matrix(dZ2);
}

/* ================= CSV ================= */

int load_csv(const char *f, Matrix **X, Matrix **Y, int *n) {
    FILE *fp = fopen(f, "r");
    if (!fp) return -1;
    char line[1024]; int c = 0;
    while (fgets(line, 1024, fp)) c++;
    rewind(fp);

    *X = allocate_matrix(c, INPUT_SIZE);
    *Y = allocate_matrix(c, OUTPUT_SIZE);
    *n = c;

    int i = 0;
    while (fgets(line, 1024, fp)) {
        char *t = strtok(line, ",");
        int j = 0;
        while (t) {
            if (j < INPUT_SIZE)
                (*X)->data[i * INPUT_SIZE + j] = atof(t);
            else
                (*Y)->data[i] = atof(t);
            t = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(fp);
    return 0;
}

/* ================= MAIN ================= */

int main() {
    srand(time(NULL));

    Matrix *X, *Y;
    int n;
    load_csv("synthetic_convex_large.csv", &X, &Y, &n);


    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    Matrix *Xb = allocate_matrix(BATCH_SIZE, INPUT_SIZE);
    Matrix *Yb = allocate_matrix(BATCH_SIZE, OUTPUT_SIZE);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    float t_forward = 0, t_activation = 0, t_output = 0, t_backprop = 0;

    for (int ep = 0; ep < EPOCHS; ep++) {
        for (int i = 0; i + BATCH_SIZE <= n; i += BATCH_SIZE) {

            memcpy(Xb->data, &X->data[i * INPUT_SIZE], BATCH_SIZE * INPUT_SIZE * sizeof(float));
            memcpy(Yb->data, &Y->data[i], BATCH_SIZE * sizeof(float));

            cudaEvent_t s1,e1,s2,e2,s3,e3,s4,e4;
            cudaEventCreate(&s1); cudaEventCreate(&e1);
            cudaEventCreate(&s2); cudaEventCreate(&e2);
            cudaEventCreate(&s3); cudaEventCreate(&e3);
            cudaEventCreate(&s4); cudaEventCreate(&e4);

            cudaEventRecord(s1);
            Matrix *Z1 = mat_mult_gpu_BT(Xb, W1);
            cudaEventRecord(e1);

            cudaEventRecord(s2);
            relu(Z1);
            cudaEventRecord(e2);

            cudaEventRecord(s3);
            Matrix *Yp = mat_mult_gpu_BT(Z1, W2);
            cudaEventRecord(e3);

            cudaEventRecord(s4);
            backpropagation(Xb, Yb, Z1, Yp, W1, W2, BATCH_SIZE);
            cudaEventRecord(e4);

            float a,b,c,d;
            cudaEventElapsedTime(&a,s1,e1);
            cudaEventElapsedTime(&b,s2,e2);
            cudaEventElapsedTime(&c,s3,e3);
            cudaEventElapsedTime(&d,s4,e4);

            t_forward += a;
            t_activation += b;
            t_output += c;
            t_backprop += d;

            free_matrix(Z1);
            free_matrix(Yp);
        }
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float total;
    cudaEventElapsedTime(&total, start, stop);

    printf("\nTemps cumulé Forward    : %f ms\n", t_forward);
    printf("Temps cumulé Activation : %f ms\n", t_activation);
    printf("Temps cumulé Output     : %f ms\n", t_output);
    printf("Temps cumulé Backprop   : %f ms\n", t_backprop);
    printf("Temps total training    : %f ms (%.4f s)\n", total, total/1000.0f);

    return 0;
}


In [ ]:
!nvcc neural_net_BT_T.cu -o neural_net_BT_T


In [ ]:
!./neural_net_BT_T


Etape 2: thread per bloc



In [ ]:
%%writefile neural_net_BT_T_T.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE 32
#define HIDDEN_SIZE 256
#define OUTPUT_SIZE 1
#define EPOCHS 100
#define LEARNING_RATE 0.01f
#define BATCH_SIZE 128
#define TILE 32  // Max threads per block (32x32=1024)

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

/* ================= MEMORY ================= */
Matrix* allocate_matrix(int r, int c) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = r;
    m->cols = c;
    m->data = (float*)malloc(r * c * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    free(m->data);
    free(m);
}

/* ================= INIT ================= */
void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = ((float)rand() / RAND_MAX * 2 - 1) * limit;
}

/* ================= TRANSPOSE ================= */
Matrix* transpose_matrix(Matrix *B) {
    Matrix *T = allocate_matrix(B->cols, B->rows);
    for (int i = 0; i < B->rows; i++)
        for (int j = 0; j < B->cols; j++)
            T->data[j * B->rows + i] = B->data[i * B->cols + j];
    return T;
}

/* ================= SHARED MEMORY KERNEL ================= */
__global__ void matmul_BT_shared(float *A, float *B_T, float *C,
                                 int A_rows, int A_cols, int B_cols) {

    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (A_cols + TILE - 1) / TILE; t++) {
        int a_col = t * TILE + threadIdx.x;
        int b_col = t * TILE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] =
            (row < A_rows && a_col < A_cols) ? A[row * A_cols + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] =
            (col < B_cols && b_col < A_cols) ? B_T[col * A_cols + b_col] : 0.0f;

        __syncthreads();
        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }

    if (row < A_rows && col < B_cols)
        C[row * B_cols + col] = sum;
}

/* ================= GPU MATMUL ================= */
Matrix* mat_mult_gpu_BT(Matrix *A, Matrix *B) {
    Matrix *B_T = transpose_matrix(B);
    Matrix *C = allocate_matrix(A->rows, B->cols);

    float *dA, *dB, *dC;
    cudaMalloc(&dA, A->rows * A->cols * sizeof(float));
    cudaMalloc(&dB, B_T->rows * B_T->cols * sizeof(float));
    cudaMalloc(&dC, C->rows * C->cols * sizeof(float));

    cudaMemcpy(dA, A->data, A->rows * A->cols * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B_T->data, B_T->rows * B_T->cols * sizeof(float), cudaMemcpyHostToDevice);

    dim3 block(TILE, TILE);
    dim3 grid((C->cols + TILE - 1) / TILE,
              (C->rows + TILE - 1) / TILE);

    matmul_BT_shared<<<grid, block>>>(dA, dB, dC, A->rows, A->cols, C->cols);
    cudaDeviceSynchronize();

    cudaMemcpy(C->data, dC, C->rows * C->cols * sizeof(float), cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free_matrix(B_T);
    return C;
}

/* ================= CPU OPS ================= */
void relu(Matrix *m) {
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = fmaxf(0.0f, m->data[i]);
}

Matrix* relu_derivative(Matrix *m) {
    Matrix *d = allocate_matrix(m->rows, m->cols);
    for (int i = 0; i < m->rows * m->cols; i++)
        d->data[i] = (m->data[i] > 0);
    return d;
}

Matrix* mat_sub(Matrix *A, Matrix *B) {
    Matrix *C = allocate_matrix(A->rows, A->cols);
    for (int i = 0; i < A->rows * A->cols; i++)
        C->data[i] = A->data[i] - B->data[i];
    return C;
}

void mat_scalar_mult(Matrix *A, float s) {
    for (int i = 0; i < A->rows * A->cols; i++)
        A->data[i] *= s;
}

/* ================= BACKPROP ================= */
void backpropagation(Matrix *X, Matrix *Y, Matrix *Z1, Matrix *Yp,
                     Matrix *W1, Matrix *W2, int bs) {

    Matrix *dZ2 = mat_sub(Yp, Y);
    mat_scalar_mult(dZ2, 2.0f / bs);

    Matrix *W2_T = transpose_matrix(W2);
    Matrix *dZ1 = mat_mult_gpu_BT(dZ2, W2_T);
    free_matrix(W2_T);

    Matrix *relu_d = relu_derivative(Z1);
    for (int i = 0; i < dZ1->rows * dZ1->cols; i++)
        dZ1->data[i] *= relu_d->data[i];
    free_matrix(relu_d);

    Matrix *X_T = transpose_matrix(X);
    free_matrix(mat_mult_gpu_BT(X_T, dZ1));
    free_matrix(X_T);
    free_matrix(dZ1);
    free_matrix(dZ2);
}

/* ================= CSV ================= */
int load_csv(const char *f, Matrix **X, Matrix **Y, int *n) {
    FILE *fp = fopen(f, "r");
    if (!fp) return -1;
    char line[1024]; int c = 0;
    while (fgets(line, 1024, fp)) c++;
    rewind(fp);

    *X = allocate_matrix(c, INPUT_SIZE);
    *Y = allocate_matrix(c, OUTPUT_SIZE);
    *n = c;

    int i = 0;
    while (fgets(line, 1024, fp)) {
        char *t = strtok(line, ",");
        int j = 0;
        while (t) {
            if (j < INPUT_SIZE)
                (*X)->data[i * INPUT_SIZE + j] = atof(t);
            else
                (*Y)->data[i] = atof(t);
            t = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(fp);
    return 0;
}

/* ================= MAIN ================= */
int main() {
    srand(time(NULL));

    Matrix *X, *Y;
    int n;
    if(load_csv("synthetic_convex_large.csv", &X, &Y, &n) != 0) {
        printf("CSV file not found\n"); return -1;
    }

    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    Matrix *Xb = allocate_matrix(BATCH_SIZE, INPUT_SIZE);
    Matrix *Yb = allocate_matrix(BATCH_SIZE, OUTPUT_SIZE);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    float t_forward = 0, t_activation = 0, t_output = 0, t_backprop = 0;

    for (int ep = 0; ep < EPOCHS; ep++) {
        for (int i = 0; i + BATCH_SIZE <= n; i += BATCH_SIZE) {
            memcpy(Xb->data, &X->data[i * INPUT_SIZE], BATCH_SIZE * INPUT_SIZE * sizeof(float));
            memcpy(Yb->data, &Y->data[i], BATCH_SIZE * sizeof(float));

            cudaEvent_t s1,e1,s2,e2,s3,e3,s4,e4;
            cudaEventCreate(&s1); cudaEventCreate(&e1);
            cudaEventCreate(&s2); cudaEventCreate(&e2);
            cudaEventCreate(&s3); cudaEventCreate(&e3);
            cudaEventCreate(&s4); cudaEventCreate(&e4);

            cudaEventRecord(s1);
            Matrix *Z1 = mat_mult_gpu_BT(Xb, W1);
            cudaEventRecord(e1);

            cudaEventRecord(s2);
            relu(Z1);
            cudaEventRecord(e2);

            cudaEventRecord(s3);
            Matrix *Yp = mat_mult_gpu_BT(Z1, W2);
            cudaEventRecord(e3);

            cudaEventRecord(s4);
            backpropagation(Xb, Yb, Z1, Yp, W1, W2, BATCH_SIZE);
            cudaEventRecord(e4);

            float a,b,c,d;
            cudaEventElapsedTime(&a,s1,e1);
            cudaEventElapsedTime(&b,s2,e2);
            cudaEventElapsedTime(&c,s3,e3);
            cudaEventElapsedTime(&d,s4,e4);

            t_forward += a;
            t_activation += b;
            t_output += c;
            t_backprop += d;

            free_matrix(Z1);
            free_matrix(Yp);
        }
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float total;
    cudaEventElapsedTime(&total, start, stop);

    printf("\nTemps cumulé Forward    : %f ms\n", t_forward);
    printf("Temps cumulé Activation : %f ms\n", t_activation);
    printf("Temps cumulé Output     : %f ms\n", t_output);
    printf("Temps cumulé Backprop   : %f ms\n", t_backprop);
    printf("Temps total training    : %f ms (%.4f s)\n", total, total/1000.0f);

    return 0;
}


In [ ]:
!nvcc neural_net_BT_T_T.cu -o neural_net_BT_T_T


In [ ]:
!./neural_net_BT_T_T

In [ ]:
# Lancez avec Nsight Systems
!nsys profile -o profile_output ./neural_net_BT_T

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!nvcc neural_net_BT_T_T.cu \
     -O3 \
     -arch=sm_75 \
     -lineinfo \
     -o neural_net_BT
!./neural_net_BT


In [ ]:
!ncu \
  --set full \
  --target-processes all \
  --force-overwrite \
  --export profile_BT \
  ./neural_net_BT


In [ ]:
from google.colab import files
files.download('profile_BT.ncu-rep')

In [ ]:
%%writefile neural_net_gpu.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE 32
#define HIDDEN_SIZE 256
#define OUTPUT_SIZE 1
#define EPOCHS 100
#define LEARNING_RATE 0.01f
#define BATCH_SIZE 128
#define TILE 32  // Max threads per block (32x32=1024)

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

/* ================= ALLOCATION ================= */
Matrix* allocate_matrix(int r, int c) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = r;
    m->cols = c;
    m->data = (float*)malloc(r * c * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    if (m) {
        free(m->data);
        free(m);
    }
}

/* ================= INIT ================= */
void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = ((float)rand() / RAND_MAX * 2 - 1) * limit;
}

/* ================= KERNELS ================= */
__global__ void matmul_BT_shared(float *A, float *B_T, float *C,
                                 int A_rows, int A_cols, int B_cols) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (A_cols + TILE - 1) / TILE; t++) {
        int a_col = t * TILE + threadIdx.x;
        int b_col = t * TILE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] =
            (row < A_rows && a_col < A_cols) ? A[row * A_cols + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] =
            (col < B_cols && b_col < A_cols) ? B_T[col * A_cols + b_col] : 0.0f;

        __syncthreads();
        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }

    if (row < A_rows && col < B_cols)
        C[row * B_cols + col] = sum;
}

__global__ void relu_kernel(float *data, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        data[idx] = fmaxf(0.0f, data[idx]);
}

__global__ void relu_derivative_kernel(float *Z, float *dZ, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        dZ[idx] *= (Z[idx] > 0) ? 1.0f : 0.0f;
}

__global__ void mat_sub_scaled_kernel(float *A, float *B, float *C,
                                      int size, float scale) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        C[idx] = (A[idx] - B[idx]) * scale;
}

__global__ void elementwise_mul_kernel(float *A, float *B, float *C, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        C[idx] = A[idx] * B[idx];
}

__global__ void update_weights_kernel(float *W, float *grad, int size, float lr) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        W[idx] -= lr * grad[idx];
}

__global__ void transpose_kernel(float *src, float *dst, int src_rows, int src_cols) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = src_rows * src_cols;

    if (idx < total) {
        int row = idx / src_cols;
        int col = idx % src_cols;
        dst[col * src_rows + row] = src[idx];
    }
}

/* ================= GPU MATMUL WITH TRANSPOSE ================= */
void mat_mult_gpu_BT(float *dA, float *dB_T, float *dC,
                     int A_rows, int A_cols, int B_cols) {
    dim3 block(TILE, TILE);
    dim3 grid((B_cols + TILE - 1) / TILE,
              (A_rows + TILE - 1) / TILE);

    matmul_BT_shared<<<grid, block>>>(dA, dB_T, dC, A_rows, A_cols, B_cols);
}

/* ================= COMPLETE GPU TRAINING ================= */
void train_neural_net_fully_gpu(Matrix *X, Matrix *Y, int n_samples) {
    // Allocate CPU matrices
    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    Matrix *W1_T = allocate_matrix(HIDDEN_SIZE, INPUT_SIZE);
    Matrix *W2_T = allocate_matrix(OUTPUT_SIZE, HIDDEN_SIZE);
    Matrix *X_batch = allocate_matrix(BATCH_SIZE, INPUT_SIZE);
    Matrix *Y_batch = allocate_matrix(BATCH_SIZE, OUTPUT_SIZE);

    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    // Transpose matrices on CPU once
    for (int i = 0; i < INPUT_SIZE; i++)
        for (int j = 0; j < HIDDEN_SIZE; j++)
            W1_T->data[j * INPUT_SIZE + i] = W1->data[i * HIDDEN_SIZE + j];

    for (int i = 0; i < HIDDEN_SIZE; i++)
        for (int j = 0; j < OUTPUT_SIZE; j++)
            W2_T->data[j * HIDDEN_SIZE + i] = W2->data[i * OUTPUT_SIZE];

    // Allocate GPU memory (persistent)
    float *d_X, *d_Y, *d_Z1, *d_A1, *d_Z2, *d_Y_pred;
    float *d_W1_T, *d_W2_T;
    float *d_dZ2, *d_dW2, *d_dZ1, *d_dW1;
    float *d_X_T, *d_A1_T;

    cudaMalloc(&d_X, BATCH_SIZE * INPUT_SIZE * sizeof(float));
    cudaMalloc(&d_Y, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_Z1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_A1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_Z2, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_Y_pred, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_W1_T, HIDDEN_SIZE * INPUT_SIZE * sizeof(float));
    cudaMalloc(&d_W2_T, OUTPUT_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_dZ2, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_dW2, HIDDEN_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_dZ1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_dW1, INPUT_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_X_T, INPUT_SIZE * BATCH_SIZE * sizeof(float));
    cudaMalloc(&d_A1_T, HIDDEN_SIZE * BATCH_SIZE * sizeof(float));

    // Copy transposed weights to GPU once
    cudaMemcpy(d_W1_T, W1_T->data, HIDDEN_SIZE * INPUT_SIZE * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_W2_T, W2_T->data, OUTPUT_SIZE * HIDDEN_SIZE * sizeof(float), cudaMemcpyHostToDevice);

    cudaEvent_t start_total, stop_total;
    cudaEventCreate(&start_total);
    cudaEventCreate(&stop_total);
    cudaEventRecord(start_total);

    float t_forward = 0, t_backprop = 0;

    // Training loop
    for (int ep = 0; ep < EPOCHS; ep++) {
        for (int i = 0; i + BATCH_SIZE <= n_samples; i += BATCH_SIZE) {
            // Prepare batch
            memcpy(X_batch->data, &X->data[i * INPUT_SIZE],
                   BATCH_SIZE * INPUT_SIZE * sizeof(float));
            memcpy(Y_batch->data, &Y->data[i],
                   BATCH_SIZE * sizeof(float));

            // Copy batch to GPU
            cudaMemcpy(d_X, X_batch->data,
                       BATCH_SIZE * INPUT_SIZE * sizeof(float),
                       cudaMemcpyHostToDevice);
            cudaMemcpy(d_Y, Y_batch->data,
                       BATCH_SIZE * OUTPUT_SIZE * sizeof(float),
                       cudaMemcpyHostToDevice);

            cudaEvent_t start_fp, stop_fp, start_bp, stop_bp;
            cudaEventCreate(&start_fp); cudaEventCreate(&stop_fp);
            cudaEventCreate(&start_bp); cudaEventCreate(&stop_bp);

            // ====== FORWARD PASS (Fully GPU) ======
            cudaEventRecord(start_fp);

            // Z1 = X * W1_T^T (using transposed weights)
            mat_mult_gpu_BT(d_X, d_W1_T, d_Z1,
                           BATCH_SIZE, INPUT_SIZE, HIDDEN_SIZE);

            // A1 = ReLU(Z1)
            int blocks = (BATCH_SIZE * HIDDEN_SIZE + 255) / 256;
            relu_kernel<<<blocks, 256>>>(d_Z1, BATCH_SIZE * HIDDEN_SIZE);
            cudaMemcpy(d_A1, d_Z1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float),
                      cudaMemcpyDeviceToDevice);

            // Z2 = A1 * W2_T^T
            mat_mult_gpu_BT(d_A1, d_W2_T, d_Z2,
                           BATCH_SIZE, HIDDEN_SIZE, OUTPUT_SIZE);

            // Y_pred = Z2 (linear output)
            cudaMemcpy(d_Y_pred, d_Z2, BATCH_SIZE * OUTPUT_SIZE * sizeof(float),
                      cudaMemcpyDeviceToDevice);

            cudaEventRecord(stop_fp);

            // ====== BACKWARD PASS (Fully GPU) ======
            cudaEventRecord(start_bp);

            // dZ2 = (Y_pred - Y) * 2/batch_size
            blocks = (BATCH_SIZE * OUTPUT_SIZE + 255) / 256;
            mat_sub_scaled_kernel<<<blocks, 256>>>(d_Y_pred, d_Y, d_dZ2,
                                                  BATCH_SIZE * OUTPUT_SIZE,
                                                  2.0f / BATCH_SIZE);

            // dW2 = A1_T * dZ2
            blocks = (BATCH_SIZE * HIDDEN_SIZE + 255) / 256;
            transpose_kernel<<<blocks, 256>>>(d_A1, d_A1_T, BATCH_SIZE, HIDDEN_SIZE);

            mat_mult_gpu_BT(d_A1_T, d_dZ2, d_dW2,
                           HIDDEN_SIZE, BATCH_SIZE, OUTPUT_SIZE);

            // Update W2_T
            blocks = (HIDDEN_SIZE * OUTPUT_SIZE + 255) / 256;
            update_weights_kernel<<<blocks, 256>>>(d_W2_T, d_dW2,
                                                  HIDDEN_SIZE * OUTPUT_SIZE,
                                                  LEARNING_RATE);

            // dZ1 = dZ2 * W2_T (using transposed weights)
            mat_mult_gpu_BT(d_dZ2, d_W2_T, d_dZ1,
                           BATCH_SIZE, OUTPUT_SIZE, HIDDEN_SIZE);

            // Apply ReLU derivative
            relu_derivative_kernel<<<blocks, 256>>>(d_Z1, d_dZ1,
                                                   BATCH_SIZE * HIDDEN_SIZE);

            // dW1 = X_T * dZ1
            blocks = (BATCH_SIZE * INPUT_SIZE + 255) / 256;
            transpose_kernel<<<blocks, 256>>>(d_X, d_X_T, BATCH_SIZE, INPUT_SIZE);

            mat_mult_gpu_BT(d_X_T, d_dZ1, d_dW1,
                           INPUT_SIZE, BATCH_SIZE, HIDDEN_SIZE);

            // Update W1_T
            blocks = (INPUT_SIZE * HIDDEN_SIZE + 255) / 256;
            update_weights_kernel<<<blocks, 256>>>(d_W1_T, d_dW1,
                                                  INPUT_SIZE * HIDDEN_SIZE,
                                                  LEARNING_RATE);

            cudaEventRecord(stop_bp);
            cudaEventSynchronize(stop_bp);

            // Accumulate timings
            float t_fp, t_bp;
            cudaEventElapsedTime(&t_fp, start_fp, stop_fp);
            cudaEventElapsedTime(&t_bp, start_bp, stop_bp);
            t_forward += t_fp;
            t_backprop += t_bp;

            cudaEventDestroy(start_fp); cudaEventDestroy(stop_fp);
            cudaEventDestroy(start_bp); cudaEventDestroy(stop_bp);
        }
    }

    // Copy final weights back to CPU
    cudaMemcpy(W1_T->data, d_W1_T, HIDDEN_SIZE * INPUT_SIZE * sizeof(float),
               cudaMemcpyDeviceToHost);
    cudaMemcpy(W2_T->data, d_W2_T, OUTPUT_SIZE * HIDDEN_SIZE * sizeof(float),
               cudaMemcpyDeviceToHost);

    // Transpose back to original format
    for (int i = 0; i < HIDDEN_SIZE; i++)
        for (int j = 0; j < INPUT_SIZE; j++)
            W1->data[j * HIDDEN_SIZE + i] = W1_T->data[i * INPUT_SIZE + j];

    for (int i = 0; i < OUTPUT_SIZE; i++)
        for (int j = 0; j < HIDDEN_SIZE; j++)
            W2->data[j * OUTPUT_SIZE + i] = W2_T->data[i * HIDDEN_SIZE + j];

    cudaEventRecord(stop_total);
    cudaEventSynchronize(stop_total);

    float total_time;
    cudaEventElapsedTime(&total_time, start_total, stop_total);

    printf("\n=== FULLY GPU OPTIMIZED ===\n");
    printf("Forward time: %.2f ms\n", t_forward);
    printf("Backprop time: %.2f ms\n", t_backprop);
    printf("Total time: %.2f ms (%.3f s)\n", total_time, total_time/1000.0f);
    printf("Speedup: %.2fx\n\n", (t_forward + t_backprop) / total_time);

    // Cleanup
    cudaFree(d_X); cudaFree(d_Y); cudaFree(d_Z1); cudaFree(d_A1);
    cudaFree(d_Z2); cudaFree(d_Y_pred); cudaFree(d_W1_T); cudaFree(d_W2_T);
    cudaFree(d_dZ2); cudaFree(d_dW2); cudaFree(d_dZ1); cudaFree(d_dW1);
    cudaFree(d_X_T); cudaFree(d_A1_T);

    cudaEventDestroy(start_total);
    cudaEventDestroy(stop_total);

    free_matrix(W1); free_matrix(W2); free_matrix(W1_T); free_matrix(W2_T);
    free_matrix(X_batch); free_matrix(Y_batch);
}

/* ================= CSV LOADING ================= */
int load_csv(const char *f, Matrix **X, Matrix **Y, int *n) {
    FILE *fp = fopen(f, "r");
    if (!fp) return -1;

    char line[1024];
    int c = 0;
    while (fgets(line, 1024, fp)) c++;
    rewind(fp);

    *X = allocate_matrix(c, INPUT_SIZE);
    *Y = allocate_matrix(c, OUTPUT_SIZE);
    *n = c;

    int i = 0;
    while (fgets(line, 1024, fp)) {
        char *t = strtok(line, ",");
        int j = 0;
        while (t) {
            if (j < INPUT_SIZE)
                (*X)->data[i * INPUT_SIZE + j] = atof(t);
            else
                (*Y)->data[i] = atof(t);
            t = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(fp);
    return 0;
}

/* ================= MAIN ================= */
int main() {
    srand(time(NULL));

    Matrix *X, *Y;
    int n;
    if (load_csv("synthetic_convex_small.csv", &X, &Y, &n) != 0) {
        printf("CSV file not found\n");
        return -1;
    }

    printf("Dataset loaded: %d samples\n", n);

    // Run fully GPU-optimized training
    train_neural_net_fully_gpu(X, Y, n);

    free_matrix(X);
    free_matrix(Y);

    return 0;
}

In [ ]:
!nvcc neural_net_gpu.cu -o neural_net_gpu

In [ ]:
!./neural_net_gpu

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!nvcc neural_net_gpu.cu \
     -O3 \
          -arch=sm_75 \
               -lineinfo \
                    -o neural_net_gpu
!./neural_net_gpu


In [ ]:
!ncu \
  --set full \
    --target-processes all \
      --profile-from-start off \
        --force-overwrite \
          --export profile_app \
            ./neural_net_gpu







In [ ]:
from google.colab import files
files.download('profile_gpu.ncu-rep')

In [ ]:
%%writefile neural_net_gpu_streams.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE 32
#define HIDDEN_SIZE 256
#define OUTPUT_SIZE 1
#define EPOCHS 100
#define LEARNING_RATE 0.01f
#define BATCH_SIZE 128
#define TILE 32

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

/* ================= ALLOCATION ================= */
Matrix* allocate_matrix(int r, int c) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = r;
    m->cols = c;
    m->data = (float*)malloc(r * c * sizeof(float));
    memset(m->data, 0, r * c * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    if (m) {
        free(m->data);
        free(m);
    }
}

/* ================= INIT ================= */
void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = ((float)rand() / RAND_MAX * 2 - 1) * limit;
}

/* ================= KERNELS ================= */
__global__ void matmul_BT_shared(float *A, float *B_T, float *C,
                                 int A_rows, int A_cols, int B_cols) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (A_cols + TILE - 1) / TILE; t++) {
        int a_col = t * TILE + threadIdx.x;
        int b_col = t * TILE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] =
            (row < A_rows && a_col < A_cols) ? A[row * A_cols + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] =
            (col < B_cols && b_col < A_cols) ? B_T[col * A_cols + b_col] : 0.0f;

        __syncthreads();
        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }

    if (row < A_rows && col < B_cols)
        C[row * B_cols + col] = sum;
}

__global__ void relu_kernel(float *input, float *output, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) output[idx] = fmaxf(0.0f, input[idx]);
}

__global__ void relu_derivative_kernel(float *grad, float *activation, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        grad[idx] = (activation[idx] > 0.0f) ? grad[idx] : 0.0f;
    }
}

__global__ void mat_sub_scaled_kernel(float *A, float *B, float *C,
                                      int size, float scale) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) C[idx] = (A[idx] - B[idx]) * scale;
}

__global__ void update_weights_kernel(float *W, float *grad, int size, float lr) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) W[idx] -= lr * grad[idx];
}

__global__ void transpose_kernel(float *src, float *dst, int r, int c) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < r * c) {
        int row = idx / c;
        int col = idx % c;
        dst[col * r + row] = src[idx];
    }
}

__global__ void mse_kernel(float *yhat, float *y, float *out, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float d = yhat[i] - y[i];
        atomicAdd(out, d*d);
    }
}

/* ================= GPU MATMUL ================= */
void mat_mult_gpu_BT(float *A, float *B_T, float *C,
                     int Ar, int Ac, int Bc, cudaStream_t s) {
    dim3 block(TILE, TILE);
    dim3 grid((Bc + TILE - 1) / TILE, (Ar + TILE - 1) / TILE);
    matmul_BT_shared<<<grid, block, 0, s>>>(A, B_T, C, Ar, Ac, Bc);
}

/* ================= TRAINING ================= */
void train_neural_net_fully_gpu(Matrix *X, Matrix *Y, int n) {

    printf("Training on %d samples\n", n);

    // ===== Normalize X and Y =====
    for(int i=0;i<n*INPUT_SIZE;i++)
        X->data[i] = (X->data[i] - 0.5f) * 2.0f; // simple normalization [-1,1]

    float y_min = Y->data[0], y_max = Y->data[0];
    for(int i=0;i<n;i++){
        if(Y->data[i]<y_min) y_min=Y->data[i];
        if(Y->data[i]>y_max) y_max=Y->data[i];
    }
    for(int i=0;i<n;i++)
        Y->data[i] = (Y->data[i]-y_min)/(y_max-y_min); // scale to [0,1]

    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    Matrix *W1_T = allocate_matrix(HIDDEN_SIZE, INPUT_SIZE);
    Matrix *W2_T = allocate_matrix(OUTPUT_SIZE, HIDDEN_SIZE);
    Matrix *Xb = allocate_matrix(BATCH_SIZE, INPUT_SIZE);
    Matrix *Yb = allocate_matrix(BATCH_SIZE, OUTPUT_SIZE);

    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    // Transpose W1 and W2 for GPU matmul
    for (int i = 0; i < INPUT_SIZE; i++)
        for (int j = 0; j < HIDDEN_SIZE; j++)
            W1_T->data[j * INPUT_SIZE + i] = W1->data[i * HIDDEN_SIZE + j];
    for (int i = 0; i < HIDDEN_SIZE; i++)
        W2_T->data[i] = W2->data[i];

    // GPU allocations
    float *d_X, *d_Y, *d_Z1, *d_A1, *d_Z2;
    float *d_W1_T, *d_W2_T;
    float *d_dZ2, *d_dW2, *d_dA1, *d_dZ1, *d_dW1;
    float *d_A1_T, *d_X_T, *d_W2, *d_mse;

    cudaMalloc(&d_X, BATCH_SIZE * INPUT_SIZE * sizeof(float));
    cudaMalloc(&d_Y, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_Z1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_A1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_Z2, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_W1_T, INPUT_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_W2_T, HIDDEN_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_dZ2, BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_dW2, HIDDEN_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_dA1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_dZ1, BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_dW1, INPUT_SIZE * HIDDEN_SIZE * sizeof(float));
    cudaMalloc(&d_A1_T, HIDDEN_SIZE * BATCH_SIZE * sizeof(float));
    cudaMalloc(&d_X_T, INPUT_SIZE * BATCH_SIZE * sizeof(float));
    cudaMalloc(&d_W2, HIDDEN_SIZE * OUTPUT_SIZE * sizeof(float));
    cudaMalloc(&d_mse, sizeof(float));

    cudaMemcpy(d_W1_T, W1_T->data, INPUT_SIZE * HIDDEN_SIZE * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_W2_T, W2_T->data, HIDDEN_SIZE * OUTPUT_SIZE * sizeof(float), cudaMemcpyHostToDevice);

    cudaStream_t s_h2d, s_fp, s_bp;
    cudaStreamCreate(&s_h2d);
    cudaStreamCreate(&s_fp);
    cudaStreamCreate(&s_bp);

    cudaEvent_t start_total, end_total;
    cudaEventCreate(&start_total); cudaEventCreate(&end_total);

    cudaEventRecord(start_total);

    for (int e = 0; e < EPOCHS; e++) {
        float epoch_mse = 0.0f;
        int num_batches = 0;

        for (int i = 0; i + BATCH_SIZE <= n; i += BATCH_SIZE) {

            memcpy(Xb->data, &X->data[i * INPUT_SIZE], BATCH_SIZE * INPUT_SIZE * sizeof(float));
            memcpy(Yb->data, &Y->data[i * OUTPUT_SIZE], BATCH_SIZE * OUTPUT_SIZE * sizeof(float));

            cudaMemcpyAsync(d_X, Xb->data, BATCH_SIZE * INPUT_SIZE * sizeof(float), cudaMemcpyHostToDevice, s_h2d);
            cudaMemcpyAsync(d_Y, Yb->data, BATCH_SIZE * OUTPUT_SIZE * sizeof(float), cudaMemcpyHostToDevice, s_h2d);
            cudaStreamSynchronize(s_h2d);

            // ===== Forward =====
            mat_mult_gpu_BT(d_X, d_W1_T, d_Z1, BATCH_SIZE, INPUT_SIZE, HIDDEN_SIZE, s_fp);
            relu_kernel<<<(BATCH_SIZE*HIDDEN_SIZE+255)/256,256,0,s_fp>>>(d_Z1,d_A1,BATCH_SIZE*HIDDEN_SIZE);
            mat_mult_gpu_BT(d_A1,d_W2_T,d_Z2,BATCH_SIZE,HIDDEN_SIZE,OUTPUT_SIZE,s_fp);

            // ===== MSE =====
            cudaMemsetAsync(d_mse,0,sizeof(float),s_fp);
            mse_kernel<<<(BATCH_SIZE+255)/256,256,0,s_fp>>>(d_Z2,d_Y,d_mse,BATCH_SIZE);
            float batch_mse;
            cudaMemcpyAsync(&batch_mse,d_mse,sizeof(float),cudaMemcpyDeviceToHost,s_fp);
            cudaStreamSynchronize(s_fp);
            epoch_mse += batch_mse / BATCH_SIZE;
            num_batches++;

            // ===== Backprop =====
            mat_sub_scaled_kernel<<<(BATCH_SIZE+255)/256,256,0,s_bp>>>(d_Z2,d_Y,d_dZ2,BATCH_SIZE,2.0f/BATCH_SIZE);

            transpose_kernel<<<(BATCH_SIZE*HIDDEN_SIZE+255)/256,256,0,s_bp>>>(d_A1,d_A1_T,BATCH_SIZE,HIDDEN_SIZE);
            mat_mult_gpu_BT(d_A1_T,d_dZ2,d_dW2,HIDDEN_SIZE,BATCH_SIZE,OUTPUT_SIZE,s_bp);
            update_weights_kernel<<<(HIDDEN_SIZE*OUTPUT_SIZE+255)/256,256,0,s_bp>>>(d_W2_T,d_dW2,HIDDEN_SIZE*OUTPUT_SIZE,LEARNING_RATE);

            transpose_kernel<<<(HIDDEN_SIZE*OUTPUT_SIZE+255)/256,256,0,s_bp>>>(d_W2_T,d_W2,HIDDEN_SIZE,OUTPUT_SIZE);
            mat_mult_gpu_BT(d_dZ2,d_W2,d_dA1,BATCH_SIZE,OUTPUT_SIZE,HIDDEN_SIZE,s_bp);

            cudaMemcpyAsync(d_dZ1,d_dA1,BATCH_SIZE*HIDDEN_SIZE*sizeof(float),cudaMemcpyDeviceToDevice,s_bp);
            relu_derivative_kernel<<<(BATCH_SIZE*HIDDEN_SIZE+255)/256,256,0,s_bp>>>(d_dZ1,d_A1,BATCH_SIZE*HIDDEN_SIZE);

            transpose_kernel<<<(BATCH_SIZE*INPUT_SIZE+255)/256,256,0,s_bp>>>(d_X,d_X_T,BATCH_SIZE,INPUT_SIZE);
            mat_mult_gpu_BT(d_X_T,d_dZ1,d_dW1,INPUT_SIZE,BATCH_SIZE,HIDDEN_SIZE,s_bp);
            update_weights_kernel<<<(INPUT_SIZE*HIDDEN_SIZE+255)/256,256,0,s_bp>>>(d_W1_T,d_dW1,INPUT_SIZE*HIDDEN_SIZE,LEARNING_RATE);

            cudaStreamSynchronize(s_bp);
        }

        epoch_mse /= num_batches;
        if (e%10==0 || e==EPOCHS-1) printf("Epoch %3d | MSE: %.6f\n", e+1,epoch_mse);
    }

    cudaEventRecord(end_total);
    cudaEventSynchronize(end_total);
    float total_ms;
    cudaEventElapsedTime(&total_ms,start_total,end_total);
    printf("Total training time: %.2f ms (%.3f s)\n", total_ms,total_ms/1000.0f);

    // ===== Cleanup =====
    cudaStreamDestroy(s_h2d); cudaStreamDestroy(s_fp); cudaStreamDestroy(s_bp);
    cudaEventDestroy(start_total); cudaEventDestroy(end_total);
    cudaFree(d_X); cudaFree(d_Y); cudaFree(d_Z1); cudaFree(d_A1); cudaFree(d_Z2);
    cudaFree(d_W1_T); cudaFree(d_W2_T);
    cudaFree(d_dZ2); cudaFree(d_dW2); cudaFree(d_dA1); cudaFree(d_dZ1); cudaFree(d_dW1);
    cudaFree(d_A1_T); cudaFree(d_X_T); cudaFree(d_W2); cudaFree(d_mse);

    free_matrix(W1); free_matrix(W2); free_matrix(W1_T); free_matrix(W2_T);
    free_matrix(Xb); free_matrix(Yb);
}

/* ================= CSV LOADING ================= */
int load_csv(const char *f, Matrix **X, Matrix **Y, int *n) {
    FILE *fp = fopen(f, "r");
    if (!fp) return -1;

    char line[1024];
    int c = 0;
    while (fgets(line, 1024, fp)) c++;
    rewind(fp);

    *X = allocate_matrix(c, INPUT_SIZE);
    *Y = allocate_matrix(c, OUTPUT_SIZE);
    *n = c;

    int i = 0;
    while (fgets(line, 1024, fp)) {
        char *t = strtok(line, ",");
        int j = 0;
        while (t) {
            if (j < INPUT_SIZE) (*X)->data[i * INPUT_SIZE + j] = atof(t);
            else (*Y)->data[i] = atof(t);
            t = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(fp);
    return 0;
}

int main() {
    srand(time(NULL));
    Matrix *X, *Y;
    int n;

    if (load_csv("synthetic_convex_small.csv", &X, &Y, &n) != 0) {
        printf("CSV not found\n");
        return -1;
    }

    train_neural_net_fully_gpu(X, Y, n);

    free_matrix(X);
    free_matrix(Y);
    return 0;
}


In [ ]:
!nvcc neural_net_gpu_streams.cu -O3 -o neural_net_gpu_streams

In [ ]:
!./neural_net_gpu_streams

In [ ]:
# 2. Ou utiliser ce code CUDA
%%writefile check_concurrency.cu
#include <cuda_runtime.h>
#include <stdio.h>

int main() {
    int device_count;
    cudaGetDeviceCount(&device_count);

    for (int i = 0; i < device_count; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);

        printf("GPU %d: %s\n", i, prop.name);
        printf("  Compute Capability: %d.%d\n", prop.major, prop.minor);
        printf("  Max concurrent kernels: %d\n", prop.concurrentKernels);
        printf("  Async engine count: %d\n", prop.asyncEngineCount);
        printf("  Can overlap kernels and memcpy: %s\n",
               prop.deviceOverlap ? "Yes" : "No");
        printf("  Can execute kernels concurrently: %s\n",
               prop.concurrentKernels ? "Yes" : "No");

        // Test pratique
        cudaStream_t streams[4];
        for (int j = 0; j < 4; j++) {
            cudaStreamCreate(&streams[j]);
        }
        printf("  Created %d streams successfully\n", 4);

        for (int j = 0; j < 4; j++) {
            cudaStreamDestroy(streams[j]);
        }
    }
    return 0;
}



In [ ]:
!nvcc check_concurrency.cu -o check_concurrency


In [ ]:
!./check_concurrency

In [ ]:
%%writefile neural_net_gpu_streams_simple.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define INPUT_SIZE 32
#define HIDDEN_SIZE 256
#define OUTPUT_SIZE 1
#define EPOCHS 10  // Réduit pour tests
#define LEARNING_RATE 0.01f
#define BATCH_SIZE 128
#define TILE 32
#define NUM_STREAMS 3  // 3 streams pour 3-way concurrency

typedef struct {
    int rows;
    int cols;
    float *data;
} Matrix;

/* ================= ALLOCATION ================= */
Matrix* allocate_matrix(int r, int c) {
    Matrix *m = (Matrix*)malloc(sizeof(Matrix));
    m->rows = r;
    m->cols = c;
    m->data = (float*)malloc(r * c * sizeof(float));
    return m;
}

void free_matrix(Matrix *m) {
    if (m) {
        free(m->data);
        free(m);
    }
}

/* ================= INIT ================= */
void xavier_init(Matrix *m, int fan_in) {
    float limit = sqrtf(6.0f / fan_in);
    for (int i = 0; i < m->rows * m->cols; i++)
        m->data[i] = ((float)rand() / RAND_MAX * 2 - 1) * limit;
}

/* ================= KERNELS ================= */
__global__ void matmul_BT_shared(float *A, float *B_T, float *C,
                                 int A_rows, int A_cols, int B_cols) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (A_cols + TILE - 1) / TILE; t++) {
        int a_col = t * TILE + threadIdx.x;
        int b_col = t * TILE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] =
            (row < A_rows && a_col < A_cols) ? A[row * A_cols + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] =
            (col < B_cols && b_col < A_cols) ? B_T[col * A_cols + b_col] : 0.0f;

        __syncthreads();
        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }

    if (row < A_rows && col < B_cols)
        C[row * B_cols + col] = sum;
}

__global__ void relu_kernel(float *data, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        data[idx] = fmaxf(0.0f, data[idx]);
}

__global__ void relu_derivative_kernel(float *Z, float *dZ, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        dZ[idx] *= (Z[idx] > 0) ? 1.0f : 0.0f;
}

__global__ void mat_sub_scaled_kernel(float *A, float *B, float *C,
                                      int size, float scale) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        C[idx] = (A[idx] - B[idx]) * scale;
}

__global__ void update_weights_kernel(float *W, float *grad, int size, float lr) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        W[idx] -= lr * grad[idx];
}

__global__ void transpose_kernel(float *src, float *dst, int src_rows, int src_cols) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = src_rows * src_cols;

    if (idx < total) {
        int row = idx / src_cols;
        int col = idx % src_cols;
        dst[col * src_rows + row] = src[idx];
    }
}

/* ================= GPU MATMUL WITH STREAM ================= */
void mat_mult_gpu_BT_stream(float *dA, float *dB_T, float *dC,
                            int A_rows, int A_cols, int B_cols,
                            cudaStream_t stream) {
    dim3 block(TILE, TILE);
    dim3 grid((B_cols + TILE - 1) / TILE,
              (A_rows + TILE - 1) / TILE);
    matmul_BT_shared<<<grid, block, 0, stream>>>(dA, dB_T, dC, A_rows, A_cols, B_cols);
}

/* ================= TRAINING AVEC 3 STREAMS ================= */
void train_neural_net_streams(Matrix *X, Matrix *Y, int n_samples) {
    printf("=== GPU AVEC %d STREAMS (3-way concurrency) ===\n", NUM_STREAMS);

    // Allocate CPU matrices
    Matrix *W1 = allocate_matrix(INPUT_SIZE, HIDDEN_SIZE);
    Matrix *W2 = allocate_matrix(HIDDEN_SIZE, OUTPUT_SIZE);
    Matrix *W1_T = allocate_matrix(HIDDEN_SIZE, INPUT_SIZE);
    Matrix *W2_T = allocate_matrix(OUTPUT_SIZE, HIDDEN_SIZE);
    Matrix *X_batch[NUM_STREAMS];
    Matrix *Y_batch[NUM_STREAMS];

    for (int s = 0; s < NUM_STREAMS; s++) {
        X_batch[s] = allocate_matrix(BATCH_SIZE, INPUT_SIZE);
        Y_batch[s] = allocate_matrix(BATCH_SIZE, OUTPUT_SIZE);
    }

    xavier_init(W1, INPUT_SIZE);
    xavier_init(W2, HIDDEN_SIZE);

    // Transpose matrices on CPU once
    for (int i = 0; i < INPUT_SIZE; i++)
        for (int j = 0; j < HIDDEN_SIZE; j++)
            W1_T->data[j * INPUT_SIZE + i] = W1->data[i * HIDDEN_SIZE + j];

    for (int i = 0; i < HIDDEN_SIZE; i++)
        for (int j = 0; j < OUTPUT_SIZE; j++)
            W2_T->data[j * HIDDEN_SIZE + i] = W2->data[i * OUTPUT_SIZE];

    // Create streams
    cudaStream_t streams[NUM_STREAMS];
    for (int i = 0; i < NUM_STREAMS; i++) {
        cudaStreamCreate(&streams[i]);
    }

    // Allocate GPU memory per stream
    float *d_X[NUM_STREAMS], *d_Y[NUM_STREAMS];
    float *d_Z1[NUM_STREAMS], *d_A1[NUM_STREAMS], *d_Z2[NUM_STREAMS], *d_Y_pred[NUM_STREAMS];
    float *d_dZ2[NUM_STREAMS], *d_dW2[NUM_STREAMS], *d_dZ1[NUM_STREAMS], *d_dW1[NUM_STREAMS];
    float *d_X_T[NUM_STREAMS], *d_A1_T[NUM_STREAMS];

    // Shared weights
    float *d_W1_T, *d_W2_T;

    for (int s = 0; s < NUM_STREAMS; s++) {
        cudaMalloc(&d_X[s], BATCH_SIZE * INPUT_SIZE * sizeof(float));
        cudaMalloc(&d_Y[s], BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
        cudaMalloc(&d_Z1[s], BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
        cudaMalloc(&d_A1[s], BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
        cudaMalloc(&d_Z2[s], BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
        cudaMalloc(&d_Y_pred[s], BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
        cudaMalloc(&d_dZ2[s], BATCH_SIZE * OUTPUT_SIZE * sizeof(float));
        cudaMalloc(&d_dW2[s], HIDDEN_SIZE * OUTPUT_SIZE * sizeof(float));
        cudaMalloc(&d_dZ1[s], BATCH_SIZE * HIDDEN_SIZE * sizeof(float));
        cudaMalloc(&d_dW1[s], INPUT_SIZE * HIDDEN_SIZE * sizeof(float));
        cudaMalloc(&d_X_T[s], INPUT_SIZE * BATCH_SIZE * sizeof(float));
        cudaMalloc(&d_A1_T[s], HIDDEN_SIZE * BATCH_SIZE * sizeof(float));
    }

    cudaMalloc(&d_W1_T, HIDDEN_SIZE * INPUT_SIZE * sizeof(float));
    cudaMalloc(&d_W2_T, OUTPUT_SIZE * HIDDEN_SIZE * sizeof(float));

    // Copy transposed weights to GPU once
    cudaMemcpy(d_W1_T, W1_T->data, HIDDEN_SIZE * INPUT_SIZE * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_W2_T, W2_T->data, OUTPUT_SIZE * HIDDEN_SIZE * sizeof(float), cudaMemcpyHostToDevice);

    cudaEvent_t start_total, stop_total;
    cudaEventCreate(&start_total);
    cudaEventCreate(&stop_total);
    cudaEventRecord(start_total);

    float t_forward = 0, t_backprop = 0;
    int n_batches = n_samples / BATCH_SIZE;

    // Training loop avec streams
    for (int ep = 0; ep < EPOCHS; ep++) {
        // Traiter les batchs en groupes de NUM_STREAMS
        for (int batch_start = 0; batch_start < n_batches; batch_start += NUM_STREAMS) {

            // Lancer NUM_STREAMS batchs en parallèle
            for (int s = 0; s < NUM_STREAMS; s++) {
                int batch_idx = batch_start + s;
                if (batch_idx >= n_batches) break;

                cudaStream_t stream = streams[s];

                // Préparer batch
                int data_start = batch_idx * BATCH_SIZE;
                memcpy(X_batch[s]->data, &X->data[data_start * INPUT_SIZE],
                       BATCH_SIZE * INPUT_SIZE * sizeof(float));
                memcpy(Y_batch[s]->data, &Y->data[data_start],
                       BATCH_SIZE * sizeof(float));

                // Copier batch vers GPU (asynchrone)
                cudaMemcpyAsync(d_X[s], X_batch[s]->data,
                               BATCH_SIZE * INPUT_SIZE * sizeof(float),
                               cudaMemcpyHostToDevice, stream);
                cudaMemcpyAsync(d_Y[s], Y_batch[s]->data,
                               BATCH_SIZE * OUTPUT_SIZE * sizeof(float),
                               cudaMemcpyHostToDevice, stream);

                cudaEvent_t start_fp, stop_fp, start_bp, stop_bp;
                cudaEventCreate(&start_fp); cudaEventCreate(&stop_fp);
                cudaEventCreate(&start_bp); cudaEventCreate(&stop_bp);

                // ====== FORWARD PASS ======
                cudaEventRecord(start_fp, stream);

                // Z1 = X * W1_T^T
                mat_mult_gpu_BT_stream(d_X[s], d_W1_T, d_Z1[s],
                                      BATCH_SIZE, INPUT_SIZE, HIDDEN_SIZE, stream);

                // A1 = ReLU(Z1)
                int blocks = (BATCH_SIZE * HIDDEN_SIZE + 255) / 256;
                relu_kernel<<<blocks, 256, 0, stream>>>(d_Z1[s], BATCH_SIZE * HIDDEN_SIZE);
                cudaMemcpyAsync(d_A1[s], d_Z1[s], BATCH_SIZE * HIDDEN_SIZE * sizeof(float),
                              cudaMemcpyDeviceToDevice, stream);

                // Z2 = A1 * W2_T^T
                mat_mult_gpu_BT_stream(d_A1[s], d_W2_T, d_Z2[s],
                                      BATCH_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, stream);

                // Y_pred = Z2
                cudaMemcpyAsync(d_Y_pred[s], d_Z2[s], BATCH_SIZE * OUTPUT_SIZE * sizeof(float),
                              cudaMemcpyDeviceToDevice, stream);

                cudaEventRecord(stop_fp, stream);

                // ====== BACKWARD PASS ======
                cudaEventRecord(start_bp, stream);

                // dZ2 = (Y_pred - Y) * 2/batch_size
                blocks = (BATCH_SIZE * OUTPUT_SIZE + 255) / 256;
                mat_sub_scaled_kernel<<<blocks, 256, 0, stream>>>(d_Y_pred[s], d_Y[s], d_dZ2[s],
                                                                BATCH_SIZE * OUTPUT_SIZE,
                                                                2.0f / BATCH_SIZE);

                // dW2 = A1_T * dZ2
                blocks = (BATCH_SIZE * HIDDEN_SIZE + 255) / 256;
                transpose_kernel<<<blocks, 256, 0, stream>>>(d_A1[s], d_A1_T[s], BATCH_SIZE, HIDDEN_SIZE);

                mat_mult_gpu_BT_stream(d_A1_T[s], d_dZ2[s], d_dW2[s],
                                      HIDDEN_SIZE, BATCH_SIZE, OUTPUT_SIZE, stream);

                // Update W2_T
                blocks = (HIDDEN_SIZE * OUTPUT_SIZE + 255) / 256;
                update_weights_kernel<<<blocks, 256, 0, stream>>>(d_W2_T, d_dW2[s],
                                                                HIDDEN_SIZE * OUTPUT_SIZE,
                                                                LEARNING_RATE);

                // dZ1 = dZ2 * W2_T
                mat_mult_gpu_BT_stream(d_dZ2[s], d_W2_T, d_dZ1[s],
                                      BATCH_SIZE, OUTPUT_SIZE, HIDDEN_SIZE, stream);

                // Apply ReLU derivative
                relu_derivative_kernel<<<blocks, 256, 0, stream>>>(d_Z1[s], d_dZ1[s],
                                                                 BATCH_SIZE * HIDDEN_SIZE);

                // dW1 = X_T * dZ1
                blocks = (BATCH_SIZE * INPUT_SIZE + 255) / 256;
                transpose_kernel<<<blocks, 256, 0, stream>>>(d_X[s], d_X_T[s], BATCH_SIZE, INPUT_SIZE);

                mat_mult_gpu_BT_stream(d_X_T[s], d_dZ1[s], d_dW1[s],
                                      INPUT_SIZE, BATCH_SIZE, HIDDEN_SIZE, stream);

                // Update W1_T
                blocks = (INPUT_SIZE * HIDDEN_SIZE + 255) / 256;
                update_weights_kernel<<<blocks, 256, 0, stream>>>(d_W1_T, d_dW1[s],
                                                                INPUT_SIZE * HIDDEN_SIZE,
                                                                LEARNING_RATE);

                cudaEventRecord(stop_bp, stream);

                // Cleanup events
                cudaEventDestroy(start_fp); cudaEventDestroy(stop_fp);
                cudaEventDestroy(start_bp); cudaEventDestroy(stop_bp);
            }

            // Synchroniser tous les streams avant le prochain groupe
            for (int s = 0; s < NUM_STREAMS; s++) {
                if (batch_start + s < n_batches) {
                    cudaStreamSynchronize(streams[s]);
                }
            }
        }
    }

    // Copy final weights back to CPU
    cudaMemcpy(W1_T->data, d_W1_T, HIDDEN_SIZE * INPUT_SIZE * sizeof(float),
               cudaMemcpyDeviceToHost);
    cudaMemcpy(W2_T->data, d_W2_T, OUTPUT_SIZE * HIDDEN_SIZE * sizeof(float),
               cudaMemcpyDeviceToHost);

    // Transpose back to original format
    for (int i = 0; i < HIDDEN_SIZE; i++)
        for (int j = 0; j < INPUT_SIZE; j++)
            W1->data[j * HIDDEN_SIZE + i] = W1_T->data[i * INPUT_SIZE + j];

    for (int i = 0; i < OUTPUT_SIZE; i++)
        for (int j = 0; j < HIDDEN_SIZE; j++)
            W2->data[j * OUTPUT_SIZE + i] = W2_T->data[i * HIDDEN_SIZE + j];

    cudaEventRecord(stop_total);
    cudaEventSynchronize(stop_total);

    float total_time;
    cudaEventElapsedTime(&total_time, start_total, stop_total);

    printf("Total time: %.2f ms (%.3f s)\n", total_time, total_time/1000.0f);

    // Cleanup
    for (int s = 0; s < NUM_STREAMS; s++) {
        cudaFree(d_X[s]); cudaFree(d_Y[s]); cudaFree(d_Z1[s]); cudaFree(d_A1[s]);
        cudaFree(d_Z2[s]); cudaFree(d_Y_pred[s]); cudaFree(d_dZ2[s]); cudaFree(d_dW2[s]);
        cudaFree(d_dZ1[s]); cudaFree(d_dW1[s]); cudaFree(d_X_T[s]); cudaFree(d_A1_T[s]);
        cudaStreamDestroy(streams[s]);
        free_matrix(X_batch[s]); free_matrix(Y_batch[s]);
    }

    cudaFree(d_W1_T); cudaFree(d_W2_T);

    cudaEventDestroy(start_total);
    cudaEventDestroy(stop_total);

    free_matrix(W1); free_matrix(W2); free_matrix(W1_T); free_matrix(W2_T);
}

/* ================= CSV LOADING ================= */
int load_csv(const char *f, Matrix **X, Matrix **Y, int *n) {
    FILE *fp = fopen(f, "r");
    if (!fp) return -1;

    char line[1024];
    int c = 0;
    while (fgets(line, 1024, fp)) c++;
    rewind(fp);

    *X = allocate_matrix(c, INPUT_SIZE);
    *Y = allocate_matrix(c, OUTPUT_SIZE);
    *n = c;

    int i = 0;
    while (fgets(line, 1024, fp)) {
        char *t = strtok(line, ",");
        int j = 0;
        while (t) {
            if (j < INPUT_SIZE)
                (*X)->data[i * INPUT_SIZE + j] = atof(t);
            else
                (*Y)->data[i] = atof(t);
            t = strtok(NULL, ",");
            j++;
        }
        i++;
    }
    fclose(fp);
    return 0;
}

/* ================= MAIN ================= */
int main() {
    srand(time(NULL));

    Matrix *X, *Y;
    int n;
    if (load_csv("synthetic_convex_large.csv", &X, &Y, &n) != 0) {
        printf("CSV file not found\n");
        return -1;
    }

    printf("Dataset loaded: %d samples\n", n);
    printf("Using %d CUDA streams for pipelining\n", NUM_STREAMS);

    // Run stream-based training
    train_neural_net_streams(X, Y, n);

    free_matrix(X);
    free_matrix(Y);

    return 0;
}

In [ ]:
!nvcc neural_net_gpu_streams_simple.cu -o neural_net_streams -O3

In [ ]:
!./neural_net_streams

In [ ]:
!nvprof --print-gpu-trace ./neural_net_streams 2>&1 | tee timeline_streams.txt


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches


# Lire le fichier gpu_trace.txt
with open('gpu_trace.txt', 'r') as f:
    lines = f.readlines()


# Parser les kernels avec leur temps réel
kernels = []
current_time_offset = None


for line in lines:
    if 'Start' in line and 'Duration' in line and 'Grid Size' in line:
        continue  # Skip header

    parts = line.strip().split()
    if len(parts) >= 12:
        try:
            start_ms = float(parts[0].replace('ms', ''))
            duration_ms = float(parts[1].replace('us', '')) / 1000.0 if 'us' in parts[1] else float(parts[1].replace('ms', ''))
            kernel_name = ' '.join(parts[11:]) if len(parts) > 11 else 'unknown'

            # Ignorer les copies mémoire
            if 'memcpy' in kernel_name or 'CUDA memcpy' in kernel_name:
                continue

            # Premier timestamp comme référence
            if current_time_offset is None:
                current_time_offset = start_ms

            kernels.append({
                'name': kernel_name,
                'start': start_ms - current_time_offset,
                'duration': duration_ms,
                'end': (start_ms - current_time_offset) + duration_ms
            })
        except:
            continue


print(f"✅ {len(kernels)} kernels CUDA détectés")


# Identifier les batchs (recherche des motifs de batch)
batch_kernels = []
current_batch = []
batch_count = 0


for i, kernel in enumerate(kernels):
    current_batch.append(kernel)

    # Un batch se termine quand on voit update_weights_kernel
    if 'update_weights_kernel' in kernel['name']:
        if len(current_batch) >= 4:  # Un batch valide a au moins 4 kernels
            batch_kernels.append(current_batch.copy())
            batch_count += 1
            current_batch = []

            if batch_count >= 4:  # On veut seulement 4 batchs
                break


# Prendre les 4 premiers batchs
if len(batch_kernels) >= 4:
    batches = batch_kernels[:4]

    # Mapper chaque type de kernel à une couleur
    kernel_colors = {}
    color_palette = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
        '#9467bd', '#8c564b', '#e377c2', '#17becf',
        '#bcbd22', '#9edae5'
    ]

    all_kernels = []
    for batch in batches:
        all_kernels.extend(batch)

    for kernel in all_kernels:
        # Nom simplifié
        if 'matmul_BT_shared' in kernel['name']:
            short_name = 'matmul_BT_shared'
        elif 'relu_kernel' in kernel['name']:
            short_name = 'relu_kernel'
        elif 'mat_sub_scaled_kernel' in kernel['name']:
            short_name = 'mat_sub_scaled'
        elif 'transpose_kernel' in kernel['name']:
            short_name = 'transpose_kernel'
        elif 'update_weights_kernel' in kernel['name']:
            short_name = 'update_weights'
        elif 'relu_derivative_kernel' in kernel['name']:
            short_name = 'relu_derivative'
        else:
            short_name = kernel['name'].split('(')[0]

        kernel['short_name'] = short_name

        if short_name not in kernel_colors:
            kernel_colors[short_name] = color_palette[len(kernel_colors) % len(color_palette)]

    print(f"📦 {len(batches)} batchs détectés")
    for i, batch in enumerate(batches):
        print(f"   Batch {i+1}: {len(batch)} kernels")

    # Créer la timeline pour 4 batchs
    fig, ax = plt.subplots(figsize=(16, 8))

    # Positions Y pour chaque batch
    y_positions = [0.85, 0.65, 0.45, 0.25]
    bar_height = 0.15

    # Tracer chaque batch
    for batch_idx, batch in enumerate(batches):
        y_pos = y_positions[batch_idx]

        for kernel in batch:
            color = kernel_colors[kernel['short_name']]
            ax.barh(y_pos, kernel['duration'], left=kernel['start'],
                    height=bar_height, color=color, alpha=0.85,
                    edgecolor='black', linewidth=0.8)

    # Configuration des axes
    ax.set_yticks(y_positions)
    ax.set_yticklabels([f'Batch {i+1}' for i in range(len(batches))],
                       fontsize=12, fontweight='bold')
    ax.set_ylim(0.1, 1.0)

    # Calculer les limites X
    all_starts = []
    all_ends = []
    for batch in batches:
        all_starts.extend([k['start'] for k in batch])
        all_ends.extend([k['end'] for k in batch])

    x_min = min(all_starts)
    x_max = max(all_ends)

    ax.set_xlim(x_min - 0.05, x_max + 0.05)
    ax.set_xlabel('Temps (ms)', fontsize=13, fontweight='bold')

    # Analyser le pipelining entre batchs
    overlaps = []
    for i in range(len(batches)-1):
        batch_i_end = batches[i][-1]['end']
        batch_j_start = batches[i+1][0]['start']

        if batch_j_start < batch_i_end:
            overlaps.append({
                'between': f"{i+1}→{i+2}",
                'overlap': batch_i_end - batch_j_start,
                'pipelined': True
            })
        else:
            overlaps.append({
                'between': f"{i+1}→{i+2}",
                'overlap': 0,
                'pipelined': False,
                'wait': batch_j_start - batch_i_end
            })

    # Déterminer le type d'exécution global
    pipelined_count = sum(1 for o in overlaps if o['pipelined'])

    if pipelined_count == 3:
        execution_type = "COMPLÈTEMENT PIPELINÉ"
        title_color = 'darkgreen'
    elif pipelined_count > 0:
        execution_type = "PARTIELLEMENT PIPELINÉ"
        title_color = 'orange'
    else:
        execution_type = "COMPLÈTEMENT SÉQUENTIEL"
        title_color = 'darkred'

    # Titre
    title = f'Exécution des 4 premiers batchs - {execution_type}'
    ax.set_title(title, fontsize=16, fontweight='bold', pad=25, color=title_color)

    # Légende détaillée
    legend_elements = []
    for kernel_name, color in kernel_colors.items():
        # Vérifier si ce kernel existe dans les batchs
        exists = any(k['short_name'] == kernel_name for k in all_kernels)
        if exists:
            legend_elements.append(mpatches.Patch(color=color, alpha=0.85, label=kernel_name))

    ax.legend(handles=legend_elements, loc='upper right', fontsize=10,
              title="Types de kernels", title_fontsize=11,
              framealpha=0.95, fancybox=True)

    # Grille subtile
    ax.grid(True, axis='x', alpha=0.15, linestyle='--', linewidth=0.5)

    # Ajouter des lignes de séparation entre batchs
    for y in [0.775, 0.575, 0.375]:
        ax.axhline(y=y, xmin=0, xmax=1, color='gray',
                   linestyle='--', linewidth=0.5, alpha=0.3)

    # Statistiques par batch
    stats_text_lines = []
    for i, batch in enumerate(batches):
        batch_start = batch[0]['start']
        batch_end = batch[-1]['end']
        batch_duration = batch_end - batch_start
        stats_text_lines.append(f"B{i+1}: {batch_duration:.2f} ms")

    stats_text = " | ".join(stats_text_lines)

    # Informations de pipelining
    pipelining_info = []
    for overlap in overlaps:
        if overlap['pipelined']:
            pipelining_info.append(f"{overlap['between']}: +{overlap['overlap']:.3f}ms")
        else:
            pipelining_info.append(f"{overlap['between']}: -{overlap['wait']:.3f}ms")

    pipelining_text = " | ".join(pipelining_info)

    # Afficher les statistiques
    ax.text(0.98, 0.02, f"{stats_text}\n{pipelining_text}",
            transform=ax.transAxes,
            ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.9))

    # Ajouter une barre d'échelle
    scale_duration = 1.0  # 1 ms
    scale_x = x_min + (x_max - x_min) * 0.02
    scale_y = 0.05

    ax.plot([scale_x, scale_x + scale_duration], [scale_y, scale_y],
            color='black', linewidth=2)
    ax.text(scale_x + scale_duration/2, scale_y - 0.01, f'{scale_duration} ms',
            ha='center', va='top', fontsize=9)

    # Sauvegarder
    plt.tight_layout()
    plt.savefig('4_batchs_timeline.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"\n💾 Timeline sauvegardée: 4_batchs_timeline.png")

    # Rapport détaillé
    print(f"\n📊 RAPPORT D'EXÉCUTION:")
    print("-" * 50)

    for i, batch in enumerate(batches):
        batch_duration = batch[-1]['end'] - batch[0]['start']
        print(f"Batch {i+1}: {len(batch)} kernels, Durée: {batch_duration:.3f} ms")

    print("-" * 50)
    print(f"\n🔍 ANALYSE DE PIPELINING:")

    total_pipelining_time = 0
    total_wait_time = 0

    for overlap in overlaps:
        if overlap['pipelined']:
            print(f"  ✅ Batchs {overlap['between']}: CHEVAUCHEMENT de {overlap['overlap']:.3f} ms")
            total_pipelining_time += overlap['overlap']
        else:
            print(f"  ❌ Batchs {overlap['between']}: ATTENTE de {overlap['wait']:.3f} ms")
            total_wait_time += overlap['wait']

    print("-" * 50)

    # Temps théorique et réel
    sequential_time = sum(batch[-1]['end'] - batch[0]['start'] for batch in batches)
    actual_time = batches[-1][-1]['end'] - batches[0][0]['start']

    if sequential_time > 0:
        efficiency = (sequential_time / actual_time) * 100
        speedup = sequential_time / actual_time

        print(f"\n📈 PERFORMANCE:")
        print(f"  Temps séquentiel théorique: {sequential_time:.3f} ms")
        print(f"  Temps réel mesuré: {actual_time:.3f} ms")
        print(f"  Gain de temps: {sequential_time - actual_time:.3f} ms")
        print(f"  Efficacité: {efficiency:.1f}%")
        print(f"  Speedup: {speedup:.2f}x")

        if pipelined_count > 0:
            print(f"\n💡 RECOMMANDATION:")
            print(f"  Votre code est {execution_type.lower()}.")
            if pipelined_count < 3:
                print(f"  Optimisation possible: Réduire les synchronisations entre batchs")
        else:
            print(f"\n⚠️  RECOMMANDATION:")
            print(f"  Votre code est entièrement séquentiel.")
            print(f"  Pensez à: Utiliser des streams CUDA différents pour chaque batch")


else:
    print(f"❌ Seulement {len(batch_kernels)} batchs détectés (4 requis)")

    # Afficher ce qu'on a trouvé
    for i, batch in enumerate(batch_kernels):
        print(f"\nBatch {i+1} ({len(batch)} kernels):")
        for k in batch[:5]:  # Afficher les 5 premiers kernels
            print(f"  • {k['name'][:40]}")
        if len(batch) > 5:
            print(f"  ... et {len(batch)-5} autres")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calcul des facteurs d'accélération
strategies = ['Baseline', 'Memory Opt', 'Full GPU', 'Streamed']
speedup_small = [1.0, 0.300/0.273, 0.300/0.100, 0.300/0.013]
speedup_medium = [1.0, 2.900/2.679, 2.900/0.176, 2.900/0.059]
speedup_large = [1.0, 29.600/28.121, 29.600/1.624, 29.600/0.542]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(strategies))
ax.plot(x, speedup_small, 'o-', linewidth=3, markersize=10,
        label='Small Dataset', color='royalblue')
ax.plot(x, speedup_medium, 's-', linewidth=3, markersize=10,
        label='Medium Dataset', color='forestgreen')
ax.plot(x[1:], speedup_large[1:], '^-', linewidth=3, markersize=10,
        label='Large Dataset', color='crimson')

# Échelle Y adaptée (commence à 0, max à 60)
ax.set_ylim(0, 60)
ax.set_yticks(np.arange(0, 61, 10))
ax.set_yticklabels([f'{i}×' for i in range(0, 61, 10)], fontsize=10)

ax.set_xlabel('Optimization Strategy', fontsize=13, fontweight='bold')
ax.set_ylabel('Speedup Factor', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=45, fontsize=11)

# Grille et légende
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=11, loc='upper left')

# Annotations détaillées
for i in range(1, len(strategies)):
    # Small
    ax.text(i, speedup_small[i] + 1, f'{speedup_small[i]:.1f}×',
            ha='center', fontsize=9, fontweight='bold', color='darkblue',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightblue', alpha=0.5))
    # Medium
    ax.text(i, speedup_medium[i] - 2, f'{speedup_medium[i]:.1f}×',
            ha='center', fontsize=9, fontweight='bold', color='darkgreen',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightgreen', alpha=0.5))
    # Large
    if i == 3:  # Streamed
        ax.text(i, speedup_large[i] - 4, f'{speedup_large[i]:.1f}×',
                ha='center', fontsize=9, fontweight='bold', color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightcoral', alpha=0.5))
    else:
        ax.text(i, speedup_large[i] + 1, f'{speedup_large[i]:.1f}×',
                ha='center', fontsize=9, fontweight='bold', color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightcoral', alpha=0.5))

# Ligne de référence
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.3, linewidth=1)

plt.tight_layout()
plt.savefig('speedup_factors.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calcul des facteurs d'accélération
strategies = ['Baseline', 'Memory Opt', 'Full GPU', 'Streamed']
speedup_small = [1.0, 0.300/0.273, 0.300/0.100, 0.300/0.013]
speedup_medium = [1.0, 2.900/2.679, 2.900/0.176, 2.900/0.059]
speedup_large = [1.0, 29.600/28.121, 29.600/1.624, 29.600/0.542]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(strategies))
ax.plot(x, speedup_small, 'o-', linewidth=3, markersize=10,
        label='Small Dataset', color='royalblue')
ax.plot(x, speedup_medium, 's-', linewidth=3, markersize=10,
        label='Medium Dataset', color='forestgreen')
ax.plot(x[1:], speedup_large[1:], '^-', linewidth=3, markersize=10,
        label='Large Dataset', color='crimson')

# Échelle Y adaptée (commence à 0, max à 60)
ax.set_ylim(0, 60)
ax.set_yticks(np.arange(0, 61, 10))
ax.set_yticklabels([f'{i}×' for i in range(0, 61, 10)], fontsize=10)

ax.set_xlabel('Optimization Strategy', fontsize=13, fontweight='bold')
ax.set_ylabel('Speedup Factor', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=45, fontsize=11)

# Grille et légende
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=11, loc='upper left')

# Annotations détaillées
for i in range(1, len(strategies)):
    # Small
    ax.text(i, speedup_small[i] + 1, f'{speedup_small[i]:.1f}×',
            ha='center', fontsize=9, fontweight='bold', color='darkblue',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightblue', alpha=0.5))
    # Medium
    ax.text(i, speedup_medium[i] - 2, f'{speedup_medium[i]:.1f}×',
            ha='center', fontsize=9, fontweight='bold', color='darkgreen',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightgreen', alpha=0.5))
    # Large
    if i == 3:  # Streamed
        ax.text(i, speedup_large[i] - 4, f'{speedup_large[i]:.1f}×',
                ha='center', fontsize=9, fontweight='bold', color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightcoral', alpha=0.5))
    else:
        ax.text(i, speedup_large[i] + 1, f'{speedup_large[i]:.1f}×',
                ha='center', fontsize=9, fontweight='bold', color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightcoral', alpha=0.5))

# Ligne de référence
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.3, linewidth=1)

plt.tight_layout()
plt.savefig('speedup_factors.png', dpi=300, bbox_inches='tight')